# 02 - RQ2 Causal Integration Latency & Velocity Friction Evaluation

### Workflow Architecture
1. **Section 1: Ingestion, Temporal Filtering, and Covariate Feature Extraction (Steps 1.1–1.3)**:
   - Load AIDev v5 (`pull_request.parquet`, `repository.parquet`, `pr_commit_details.parquet`) and MOSAIC-3M across all 6 agent sources.
   - Restrict analysis to the temporal creation cohort ($T_2$: June 1 – July 31, 2025, with `merged_at <= 2025-08-31` observation cutoff).
   - Enforce $> 100$ stars repository threshold on MOSAIC-3M for cross-dataset parity.
   - Extract `merge_latency` and functional application volume (`app_count`, `app_line_churn`).
   - Conduct construct validity audit on commit squashing lower-bound.
2. **Section 2: Contextual Pre-Match Exploratory Analysis (Architectural Complexity & Test Inclusion)**:
   - Classify architectural root directories and measure baseline latency across architectural components.
   - Extract `has_tests` verification status and assess test suite prevalence.
   - Decompose dual-scope (`App_and_Infra`) PRs into infrastructure subdomains (`CI_CD`, `Containers`, `Automation`, `Provisioning`).
3. **Section 3: Matching Strata Preparation, 1-to-1 Archetype Mapping, & Common Support Validation**:
   - Construct 1-to-1 PR mapping to architectural archetypes and verify common support.
   - Diagnostic audit on 4D strata cell occupancy and orphan treatment analysis.
4. **Section 4: Exact Multidimensional Strata Matching, Balance Verification, & Treatment Dose Audits**:
   - Execute greedy 1:1 nearest-neighbor matching within exact `(agent, architectural_archetype, has_tests, ecosystem)` strata using 1D distance matching on `z_log_app_count` with a strict 0.28 caliper.
   - Report ATT on `merge_latency` and verify covariate balance.
   - Audit treatment dose (infrastructure file distribution) and application source vs. config churn decomposition.
5. **Section 5: Multidimensional Velocity Friction Analysis (ATT, Log-Normal AFT Modeling, & Bottlenecks)**:
   - **Dimension 1 (Macro Latency Inflation)**: Wilcoxon signed-rank test (Pratt method) and Cliff's Delta effect size on matched pairs.
   - **Dimension 2 (Conditional Integration Friction)**: Log-Normal Parametric AFT regression model with Huber-White cluster-robust sandwich covariance clustered on `pair_id`.
   - **Dimension 3 (Subdomain Decomposed Survival Model)**: Additive hazard decomposition isolating CI/CD, Automation, Containers, and Provisioning differentials.
   - **Contextual Bottlenecks**: Cross-dataset infrastructure subdomain latency differentials.
6. **Section 6: Treatment Integrity & Structural Dual-Scope Diagnostics (Active Authoring & Strict vs. Weak Nuance)**:
   - Assess active authoring vs. pure deletion rates in matched treatment units.
   - Evaluate latency differences between Strict Dual-Scope (core code) vs. Weak Dual-Scope (manifest-only) changes.

In [1]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')
import re
import numpy as np
import pandas as pd
from IPython.display import display

# Ensure project root is on sys.path regardless of execution directory
current_dir = os.path.abspath(".")
if os.path.exists(os.path.join(current_dir, "src", "pipeline_utils.py")) or os.path.exists(os.path.join(current_dir, "pipeline_utils.py")):
    sys.path.insert(0, current_dir)
else:
    parent_dir = os.path.abspath("..")
    if parent_dir not in sys.path:
        sys.path.insert(0, parent_dir)

# Import modular pipeline utilities
from src.pipeline_utils import (
    ARCHETYPE_MAPPING,
    map_architectural_archetype,
    map_ecosystem,
    assign_pr_scope,
    parse_mosaic_files_array,
    perform_greedy_strata_matching,
    generate_matched_cohorts,
    CACHE_DIR,
    load_dataset_cached,
    filter_temporal_cohort,
    deduplicate_commits,
    parse_file_metadata,
    classify_file_domain,
    APP_DIRS,
)

# Display settings to ensure wide, clean tabular outputs without line-wrapping
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)

print("Environment setup and pipeline utilities successfully loaded.")


Environment setup and pipeline utilities successfully loaded.


## Section 1: Ingestion, Temporal Filtering, and Covariate Feature Extraction

### 1.1 Cohort Loading & Universal Filtering

We load both datasets and isolate the cross-validation cohort ($T_2$: June 1 – July 31, 2025 creation cohort, with `merged_at <= 2025-08-31`).
For MOSAIC-3M, we also enforce the $> 100$ stars threshold on base repositories to ensure comparability with AIDev v5.
Notice that subjective task type ingestion (`pr_task_type.parquet`) has been completely removed in accordance with the deterministic multi-dimensional matching framework.

In [2]:
print("=== 1.1 Loading Datasets and Applying Cohort Filters ===")

# 1. AIDev v5: Load pull_request table
df_aidev_pr = load_dataset_cached(
    hf_url="hf://datasets/hao-li/AIDev-7.6M/pull_request.parquet",
    filename="aidev_v5_pull_request.parquet"
)
print(f"AIDev v5 Raw PR Table: {df_aidev_pr.shape[0]:,} rows")

# Apply universal temporal cohort filter to AIDev PRs
df_aidev_pr_filtered = filter_temporal_cohort(df_aidev_pr)
print(f"AIDev v5 Temporal Filtered PRs (Jun-Aug 2025): {len(df_aidev_pr_filtered):,}")

# Load repository table to extract language and stars for repository confounding control
df_aidev_repo = load_dataset_cached(
    hf_url="hf://datasets/hao-li/AIDev-7.6M/repository.parquet",
    filename="aidev_v5_repository.parquet"
)
repo_meta_aidev = df_aidev_repo[["id", "language", "stars"]].rename(
    columns={"id": "repo_id", "language": "primary_language"}
)
df_aidev_pr_filtered = df_aidev_pr_filtered.merge(repo_meta_aidev, on="repo_id", how="left")
df_aidev_pr_filtered["primary_language"] = df_aidev_pr_filtered["primary_language"].fillna("Unknown")
df_aidev_pr_filtered["stars"] = df_aidev_pr_filtered["stars"].fillna(0)

# 2. MOSAIC-3M: Load Repositories, PullRequests, and Commits across all 6 agent sources
mosaic_agents = ["Claude", "Codex", "Copilot", "Devin", "Human", "Jules"]
repo_dfs = []
pr_dfs = []
commit_dfs = []

for agent in mosaic_agents:
    r_df = load_dataset_cached(
        hf_url=f"hf://datasets/AISE-TUDelft/MOSAIC-agentic-3m/data/{agent}/Repositories/train-00000-of-00001.parquet",
        filename=f"mosaic_repo_{agent.lower()}.parquet"
    )
    repo_dfs.append(r_df)
    
    p_df = load_dataset_cached(
        hf_url=f"hf://datasets/AISE-TUDelft/MOSAIC-agentic-3m/data/{agent}/PullRequests/train-00000-of-00001.parquet",
        filename=f"mosaic_pr_{agent.lower()}.parquet"
    )
    p_df["agent_source"] = agent
    pr_dfs.append(p_df)
    
    c_df = load_dataset_cached(
        hf_url=f"hf://datasets/AISE-TUDelft/MOSAIC-agentic-3m/data/{agent}/Commits/train-00000-of-00001.parquet",
        filename=f"mosaic_commits_{agent.lower()}.parquet",
        columns=["pr_id", "sha", "additions", "deletions", "message_headline"]
    )
    commit_dfs.append(c_df)

df_mosaic_repo = pd.concat(repo_dfs, ignore_index=True)
df_mosaic_pr = pd.concat(pr_dfs, ignore_index=True)
df_mosaic_commits = pd.concat(commit_dfs, ignore_index=True)

print(f"MOSAIC-3M Raw PR Table:      {len(df_mosaic_pr):,} rows across {len(mosaic_agents)} agent sources")
print(f"MOSAIC-3M Raw Commits Table: {len(df_mosaic_commits):,} rows")

# Enforce MOSAIC-3M repository filter (> 100 stars)
dedup_key = "name_with_owner" if "name_with_owner" in df_mosaic_repo.columns else "id"
df_mosaic_repo_unique = df_mosaic_repo.drop_duplicates(subset=[dedup_key]).copy()
df_mosaic_repo_gt100 = df_mosaic_repo_unique[df_mosaic_repo_unique["stargazer_count"] > 100].copy()
valid_mosaic_repo_ids = set(df_mosaic_repo_gt100["id"].dropna().unique())

# Apply universal temporal filter and repository star filter to MOSAIC-3M
df_mosaic_pr_temporal = filter_temporal_cohort(df_mosaic_pr)
df_mosaic_pr_temporal["base_repo_id"] = df_mosaic_pr_temporal["base_repository"].apply(
    lambda x: x.get("id") if isinstance(x, dict) else None
)
df_mosaic_pr_filtered = df_mosaic_pr_temporal[df_mosaic_pr_temporal["base_repo_id"].isin(valid_mosaic_repo_ids)].copy()

# Merge repository metadata (primary_language and stargazer_count -> stars)
repo_meta_mosaic = df_mosaic_repo_gt100[["id", "primary_language", "stargazer_count"]].rename(
    columns={"id": "base_repo_id", "stargazer_count": "stars"}
).drop_duplicates(subset=["base_repo_id"])
df_mosaic_pr_filtered = df_mosaic_pr_filtered.merge(repo_meta_mosaic, on="base_repo_id", how="left")
df_mosaic_pr_filtered["primary_language"] = df_mosaic_pr_filtered["primary_language"].fillna("Unknown")
df_mosaic_pr_filtered["stars"] = df_mosaic_pr_filtered["stars"].fillna(0)
print(f"MOSAIC-3M Filtered PRs (>100 stars & Jun-Aug 2025): {len(df_mosaic_pr_filtered):,}")

=== 1.1 Loading Datasets and Applying Cohort Filters ===


AIDev v5 Raw PR Table: 361,296 rows


AIDev v5 Temporal Filtered PRs (Jun-Aug 2025): 26,870


MOSAIC-3M Raw PR Table:      111,969 rows across 6 agent sources
MOSAIC-3M Raw Commits Table: 374,891 rows


MOSAIC-3M Filtered PRs (>100 stars & Jun-Aug 2025): 5,206


### 1.2 Feature Extraction: Latency Outcomes and Application Volume Control

To avoid **Post-Treatment Bias** in our causal inference setup, we distinguish between pre-treatment controls and post-treatment outcomes:
- **Application Volume Control (`app_count`)**: Total unique application files modified per PR (`is_app == True`). Matching on `total_files_recorded` introduces a confounding bias because App+Infra PRs naturally modify more total files than App-Only PRs of the same application scope. By matching strictly on Application File Volume (`app_count`), we compare PRs of equivalent functional development breadth. PRs with null or $\le 0$ application files (`app_count <= 0`) are dropped.
- **Total Files (`total_files_recorded`)**: Total unique files modified per PR retained as an auxiliary breadth metric.
- **Outcome 1 - Velocity (`merge_latency`)**: Elapsed time in hours from PR creation to merge event:
  $$\text{merge\_latency} = \frac{\text{merged\_at} - \text{created\_at}}{3600\,\text{seconds}}$$
- **Outcome 2 - Rework (`gross_commit_churn`)**: Total lines modified across all commits per PR (evaluated for construct validity regarding commit squashing):
  - In **AIDev v5**, aggregated from `pr_commit_details.parquet` by summing `changes` across all commits per PR.
  - In **MOSAIC-3M**, aggregated from `df_mosaic_commits` by summing `additions` and `deletions` per PR.
  - **Upstream Merge Filtering**: Merge commits (e.g., `merge branch`, `merge pull request`, `merge remote-tracking`, `automatic merge`, `auto-merge`) are heuristically filtered out to prevent upstream repository syncs from artificially inflating rework churn.

In [3]:
print("=== 1.2 Feature Extraction: Latency Outcomes & Application Volume Controls ===")

# --------------------------------------------------------------------------
# 1. AIDev v5: Calculate merge_latency, app_count, total_files_recorded, and gross_commit_churn
# --------------------------------------------------------------------------
df_aidev_pr_filtered["merged_at_dt"] = pd.to_datetime(df_aidev_pr_filtered["merged_at"], utc=True)
df_aidev_pr_filtered["created_at_dt"] = pd.to_datetime(df_aidev_pr_filtered["created_at"], utc=True)
df_aidev_pr_filtered["merge_latency"] = (
    df_aidev_pr_filtered["merged_at_dt"] - df_aidev_pr_filtered["created_at_dt"]
).dt.total_seconds() / 3600.0

# Load raw pr_commit_details and inner join on filtered PR IDs (including commit message for merge filter)
df_commit_details = load_dataset_cached(
    hf_url="hf://datasets/hao-li/AIDev-7.6M/pr_commit_details.parquet",
    filename="aidev_v5_pr_commit_details.parquet",
    columns=["pr_id", "sha", "filename", "status", "changes", "message"]
)
df_commit_joined = df_commit_details.merge(
    df_aidev_pr_filtered[["id"]], left_on="pr_id", right_on="id", how="inner"
)

# Apply classify_file_domain to determine application files (is_app)
file_classes = [
    classify_file_domain(fn, st, ch)
    for fn, st, ch in zip(df_commit_joined["filename"], df_commit_joined["status"], df_commit_joined["changes"])
]
df_commit_joined["is_app"] = [c["is_app"] for c in file_classes]

# Compute total unique files and unique application files modified per PR (Volume Control)
aidev_total = df_commit_joined.dropna(subset=["filename"]).groupby("pr_id")["filename"].nunique().reset_index(name="total_files_recorded")
aidev_app = df_commit_joined[df_commit_joined["is_app"]].dropna(subset=["filename"]).groupby("pr_id")["filename"].nunique().reset_index(name="app_count")

aidev_app_churn = df_commit_joined[df_commit_joined["is_app"]].groupby("pr_id")["changes"].sum().reset_index(name="app_line_churn")

aidev_files = aidev_total.merge(aidev_app, on="pr_id", how="left").merge(aidev_app_churn, on="pr_id", how="left")
aidev_files["app_count"] = aidev_files["app_count"].fillna(0).astype(int)
aidev_files["app_line_churn"] = aidev_files["app_line_churn"].fillna(0).astype(int)

df_aidev_pr_filtered = df_aidev_pr_filtered.merge(aidev_files, left_on="id", right_on="pr_id", how="left")
df_aidev_pr_filtered["total_files_recorded"] = df_aidev_pr_filtered["total_files_recorded"].fillna(0).astype(int)
df_aidev_pr_filtered["app_count"] = df_aidev_pr_filtered["app_count"].fillna(0).astype(int)
df_aidev_pr_filtered["app_line_churn"] = df_aidev_pr_filtered["app_line_churn"].fillna(0).astype(int)

# Group raw commits directly by pr_id to capture total iterative churn (Outcome Metric - Rework)
# Filter out merge commits to prevent upstream syncs from inflating churn
merge_keywords = ("merge branch", "merge pull request", "merge remote-tracking", "automatic merge", "auto-merge")
is_merge = df_commit_joined["message"].str.lower().str.startswith(merge_keywords, na=False)
df_commit_no_merge = df_commit_joined[~is_merge]
aidev_churn = df_commit_no_merge.groupby("pr_id")["changes"].sum().reset_index()
aidev_churn.rename(columns={"changes": "gross_commit_churn"}, inplace=True)
df_aidev_pr_filtered = df_aidev_pr_filtered.merge(aidev_churn, left_on="id", right_on="pr_id", how="left")
df_aidev_pr_filtered["gross_commit_churn"] = df_aidev_pr_filtered["gross_commit_churn"].fillna(0)

# --------------------------------------------------------------------------
# 2. MOSAIC-3M: Calculate merge_latency, app_count, total_files_recorded, and gross_commit_churn
# --------------------------------------------------------------------------
df_mosaic_pr_filtered["merged_at_dt"] = pd.to_datetime(df_mosaic_pr_filtered["merged_at"], utc=True)
df_mosaic_pr_filtered["created_at_dt"] = pd.to_datetime(df_mosaic_pr_filtered["created_at"], utc=True)
df_mosaic_pr_filtered["merge_latency"] = (
    df_mosaic_pr_filtered["merged_at_dt"] - df_mosaic_pr_filtered["created_at_dt"]
).dt.total_seconds() / 3600.0

# Compute total unique files and unique application files per PR from nested files column
mosaic_file_metrics = df_mosaic_pr_filtered["files"].apply(parse_mosaic_files_array)
df_mosaic_pr_filtered["total_files_recorded"] = mosaic_file_metrics["total_files_recorded"].astype(int)
df_mosaic_pr_filtered["app_count"] = mosaic_file_metrics["app_count"].astype(int)
df_mosaic_pr_filtered["app_line_churn"] = mosaic_file_metrics["app_line_churn"].astype(int)
df_mosaic_pr_filtered["has_tests"] = mosaic_file_metrics["has_tests"]

# Aggregate commit-level churn (additions + deletions) across all commits per PR (Outcome Metric - Rework)
df_mosaic_commits["additions"] = pd.to_numeric(df_mosaic_commits["additions"], errors="coerce").fillna(0)
df_mosaic_commits["deletions"] = pd.to_numeric(df_mosaic_commits["deletions"], errors="coerce").fillna(0)
df_mosaic_commits["churn"] = df_mosaic_commits["additions"] + df_mosaic_commits["deletions"]

is_merge = df_mosaic_commits["message_headline"].str.lower().str.startswith(merge_keywords, na=False)
df_mosaic_commits_no_merge = df_mosaic_commits[~is_merge]
mosaic_churn = df_mosaic_commits_no_merge.groupby("pr_id")["churn"].sum().reset_index()
mosaic_churn.rename(columns={"churn": "gross_commit_churn"}, inplace=True)
df_mosaic_pr_filtered = df_mosaic_pr_filtered.merge(mosaic_churn, left_on="id", right_on="pr_id", how="left")
df_mosaic_pr_filtered["gross_commit_churn"] = df_mosaic_pr_filtered["gross_commit_churn"].fillna(0)

# --------------------------------------------------------------------------
# 3. Drop PRs with null or <= 0 app_count (Application Volume Control)
# --------------------------------------------------------------------------
aidev_pre_drop = len(df_aidev_pr_filtered)
df_aidev_pr_filtered = df_aidev_pr_filtered[df_aidev_pr_filtered["app_count"] > 0].copy()
aidev_post_drop = len(df_aidev_pr_filtered)

mosaic_pre_drop = len(df_mosaic_pr_filtered)
df_mosaic_pr_filtered = df_mosaic_pr_filtered[df_mosaic_pr_filtered["app_count"] > 0].copy()
mosaic_post_drop = len(df_mosaic_pr_filtered)

print(f"AIDev v5 Filtered PRs:   {aidev_post_drop:,} retained (dropped {aidev_pre_drop - aidev_post_drop:,} with 0/null app_count)")
print(f"MOSAIC-3M Filtered PRs:  {mosaic_post_drop:,} retained (dropped {mosaic_pre_drop - mosaic_post_drop:,} with 0/null app_count)")

# --------------------------------------------------------------------------
# 4. Summary Statistics for Control Variable & Outcome Metrics
# --------------------------------------------------------------------------
feature_summary = pd.DataFrame({
    "AIDev App Files (Control)": df_aidev_pr_filtered["app_count"].describe(),
    "MOSAIC App Files (Control)": df_mosaic_pr_filtered["app_count"].describe(),
    "AIDev Total Files": df_aidev_pr_filtered["total_files_recorded"].describe(),
    "MOSAIC Total Files": df_mosaic_pr_filtered["total_files_recorded"].describe(),
    "AIDev Latency (h) [Outcome]": df_aidev_pr_filtered["merge_latency"].describe(),
    "MOSAIC Latency (h) [Outcome]": df_mosaic_pr_filtered["merge_latency"].describe(),
    "AIDev Gross Churn [Outcome]": df_aidev_pr_filtered["gross_commit_churn"].describe(),
    "MOSAIC Gross Churn [Outcome]": df_mosaic_pr_filtered["gross_commit_churn"].describe()
})
print("\n--- Summary Statistics of Volume Control & Outcome Features ---")
display(feature_summary.round(2))


=== 1.2 Feature Extraction: Latency Outcomes & Application Volume Controls ===


AIDev v5 Filtered PRs:   14,747 retained (dropped 12,123 with 0/null app_count)
MOSAIC-3M Filtered PRs:  3,648 retained (dropped 1,558 with 0/null app_count)

--- Summary Statistics of Volume Control & Outcome Features ---


,AIDev App Files (Control),MOSAIC App Files (Control),AIDev Total Files,MOSAIC Total Files,AIDev Latency (h) [Outcome],MOSAIC Latency (h) [Outcome],AIDev Gross Churn [Outcome],MOSAIC Gross Churn [Outcome]
count,14747.00,3648.00,14747.00,3648.00,14747.00,3648.00,14747.00,3648.00
mean,8.01,5.89,17.70,9.19,26.67,62.17,1411.33,3524.91
std,31.61,11.46,54.27,16.08,114.78,164.75,15515.10,37716.38
min,1.00,1.00,1.00,1.00,0.00,0.00,0.00,1.00
25%,1.00,1.00,2.00,2.00,0.01,0.41,32.00,20.00
50%,1.00,2.00,4.00,4.00,0.07,4.00,97.00,105.00
75%,4.00,5.00,10.00,9.00,1.97,41.14,303.00,488.75
max,920.00,100.00,1484.00,100.00,1970.37,1836.92,1046645.00,1366404.00


### 1.3 Construct Validity Audit: Commit Squashing & Churn Lower-Bound

Because pull request workflows frequently squash intermediate commits upon merge, `gross_commit_churn` serves as a conservative lower-bound metric of actual iterative rework. We evaluate commit count distributions across both cohorts to empirically verify single-commit prevalence.

In [4]:
print("=== 1.3 Construct Validity Audit: Commit Squashing & Churn Lower-Bound ===")

# --------------------------------------------------------------------------
# 1. Calculate Commit Counts per PR (Unique SHA Hashes per PR)
# --------------------------------------------------------------------------
# AIDev v5: Group non-merge commits by pr_id for evaluated PR cohort
aidev_commits_per_pr = df_commit_no_merge[df_commit_no_merge["pr_id"].isin(df_aidev_pr_filtered["id"])].groupby("pr_id")["sha"].nunique()

# MOSAIC-3M: Group non-merge commits by pr_id for evaluated PR cohort
df_mosaic_commits_eval = df_mosaic_commits_no_merge[df_mosaic_commits_no_merge["pr_id"].isin(df_mosaic_pr_filtered["id"])]
mosaic_commits_per_pr = df_mosaic_commits_eval.groupby("pr_id")["sha"].nunique()

# --------------------------------------------------------------------------
# 2. Generate Commit Granularity Distribution Summary
# --------------------------------------------------------------------------
def summarize_commit_granularity(commit_series: pd.Series, dataset_name: str) -> pd.Series:
    total_prs = len(commit_series)
    c1 = (commit_series == 1).sum()
    c2_5 = ((commit_series >= 2) & (commit_series <= 5)).sum()
    c6_plus = (commit_series >= 6).sum()
    
    return pd.Series({
        "Total Evaluated PRs": f"{total_prs:,}",
        "Exactly 1 Commit (Squashed / Single-Commit)": f"{c1:,} ({c1 / total_prs * 100:.2f}%)",
        "2-5 Commits (Iterative Development)": f"{c2_5:,} ({c2_5 / total_prs * 100:.2f}%)",
        "6+ Commits (High-Churn Exploration)": f"{c6_plus:,} ({c6_plus / total_prs * 100:.2f}%)",
        "Median Commits / PR": f"{commit_series.median():.1f}",
        "Mean Commits / PR": f"{commit_series.mean():.2f}",
        "Max Commits / PR": f"{commit_series.max():,}"
    }, name=dataset_name)

squash_summary = pd.DataFrame([
    summarize_commit_granularity(aidev_commits_per_pr, "AIDev v5 (Non-Merge)"),
    summarize_commit_granularity(mosaic_commits_per_pr, "MOSAIC-3M (Non-Merge)")
]).T

print("\n--- Commit Count Distribution & Squashing Audit ---")
display(squash_summary)

# Construct Validity Assessment
p1_aidev = (aidev_commits_per_pr == 1).mean() * 100
p1_mosaic = (mosaic_commits_per_pr == 1).mean() * 100
print(f"\n[Construct Validity Conclusion]:")
print(f"  - AIDev v5: {p1_aidev:.2f}% of PRs have exactly 1 commit.")
print(f"  - MOSAIC-3M: {p1_mosaic:.2f}% of PRs have exactly 1 commit.")
print(f"  - High single-commit prevalence confirms that intermediate commits are frequently squashed upon merge.")
print(f"  - Consequently, gross_commit_churn represents a CONSERVATIVE LOWER-BOUND metric of actual iterative rework.")


=== 1.3 Construct Validity Audit: Commit Squashing & Churn Lower-Bound ===

--- Commit Count Distribution & Squashing Audit ---


,AIDev v5 (Non-Merge),MOSAIC-3M (Non-Merge)
Total Evaluated PRs,"14,747","3,648"
Exactly 1 Commit (Squashed / Single-Commit),"10,119 (68.62%)","1,532 (42.00%)"
2-5 Commits (Iterative Development),"3,175 (21.53%)","1,435 (39.34%)"
6+ Commits (High-Churn Exploration),"1,453 (9.85%)",681 (18.67%)
Median Commits / PR,1.0,2.0
Mean Commits / PR,2.44,4.14
Max Commits / PR,30,100



[Construct Validity Conclusion]:
  - AIDev v5: 68.62% of PRs have exactly 1 commit.
  - MOSAIC-3M: 42.00% of PRs have exactly 1 commit.
  - High single-commit prevalence confirms that intermediate commits are frequently squashed upon merge.
  - Consequently, gross_commit_churn represents a CONSERVATIVE LOWER-BOUND metric of actual iterative rework.


## Section 2: Contextual Pre-Match Exploratory Analysis

### 2.1 Application Complexity Proxy (Root Directory vs. Latency Multipliers)

To test how structural location and architectural complexity influence infrastructure integration latency, we:
1. Deduplicate chronological commit file records via `deduplicate_commits()`.
2. Classify commit files using `classify_file_domain()` to tag PR scopes: `App_Only`, `App_and_Infra`, and `Infra_Only`.
3. Extract each touched file's `root_dir` (e.g., `src/`, `packages/`, `apps/`, `core/`) using `parse_file_metadata()`.
4. Compute the median `merge_latency` across application root directories to quantify the integration latency multiplier.

In [5]:
print("=== 2.1 Application Complexity Proxy: Root Directory vs. Merge Latency ===")

# Deduplicate chronological commit file records to retain the final state per unique file
df_commit_details["seq_order"] = df_commit_details.index
df_commit_joined = df_commit_details.merge(
    df_aidev_pr_filtered[["id"]], left_on="pr_id", right_on="id", how="inner"
)
df_commit_dedup = deduplicate_commits(df_commit_joined)

# Classify commit files to isolate App_Only vs App_and_Infra PRs
file_classes = [
    classify_file_domain(fn, st, ch)
    for fn, st, ch in zip(df_commit_dedup["filename"], df_commit_dedup["status"], df_commit_dedup["changes"])
]
df_commit_dedup["is_app"] = [c["is_app"] for c in file_classes]
df_commit_dedup["is_infra"] = [c["is_infra"] for c in file_classes]
df_commit_dedup["infra_category"] = [c["infra_category"] for c in file_classes]
df_commit_dedup["is_test"] = [c["sub_type"] == "Test" for c in file_classes]

# PR Scope Tagging
pr_scope = df_commit_dedup.groupby("pr_id").agg(
    app_files=("is_app", "sum"),
    infra_files=("is_infra", "sum")
).reset_index()

pr_scope["scope"] = pr_scope.apply(lambda x: assign_pr_scope(x['app_files'], x['infra_files']), axis=1)

# Extract test_count per PR for AIDev v5
aidev_test_count = df_commit_dedup.groupby("pr_id")["is_test"].sum().reset_index(name="test_count")
df_aidev_pr_filtered = df_aidev_pr_filtered.merge(aidev_test_count, left_on="id", right_on="pr_id", how="left")
df_aidev_pr_filtered["test_count"] = df_aidev_pr_filtered["test_count"].fillna(0).astype(int)

# Extract root_dir for each unique commit file
df_commit_dedup["root_dir"] = [parse_file_metadata(f)["root_dir"] for f in df_commit_dedup["filename"]]

# Merge scope and merge_latency to files
df_pr_files = df_commit_dedup.merge(pr_scope[["pr_id", "scope"]], on="pr_id")
df_pr_files = df_pr_files.merge(df_aidev_pr_filtered[["id", "merge_latency"]], left_on="pr_id", right_on="id")

# Filter to App_Only and App_and_Infra cohorts
df_target = df_pr_files[df_pr_files["scope"].isin(["App_Only", "App_and_Infra"])].copy()

# Deduplicate to (pr_id, root_dir) so each PR counts once per directory it touches
pr_roots = df_target.drop_duplicates(subset=["pr_id", "root_dir"])

# Target architectural application directories defined in taxonomy (APP_DIRS)
target_app_roots = [f"{d}/" for d in sorted(APP_DIRS)]
pr_app_roots = pr_roots[pr_roots["root_dir"].isin(target_app_roots)].copy()

# Grouped table calculating median merge_latency and PR count segmented by scope
complexity_table = pr_app_roots.groupby(["root_dir", "scope"])["merge_latency"].agg(
    Median_Latency_Hours="median",
    PR_Count="count"
).unstack()

# Calculate integration latency multiplier
complexity_table["Latency_Multiplier (x)"] = (
    complexity_table[("Median_Latency_Hours", "App_and_Infra")] / 
    complexity_table[("Median_Latency_Hours", "App_Only")]
).round(1)

print("\n--- Median Merge Latency (Hours) Across Application Root Directories ---")
display(complexity_table.round(2))


=== 2.1 Application Complexity Proxy: Root Directory vs. Merge Latency ===



--- Median Merge Latency (Hours) Across Application Root Directories ---


Median_Latency_Hours               PR_Count               Latency_Multiplier (x)
scope                   App_Only App_and_Infra App_Only App_and_Infra                       
root_dir                                                                                    
api/                        1.55        153.01     41.0           4.0                   98.7
app/                        1.90         64.86    304.0          29.0                   34.2
apps/                       1.76         66.52    420.0          81.0                   37.7
backend/                    0.04         10.15     66.0           5.0                  253.0
client/                    72.14        154.21     29.0           8.0                    2.1
compile/                    0.07           NaN   1181.0           NaN                    NaN
compiler/                   0.00           NaN   1016.0           NaN                    NaN
content/                   43.89         87.28     18.0           2.0                    2.0
core/                       0.22        260.09    155.0          11.0                 1200.4
crates/                     0.68         49.83    173.0          19.0                   73.8
examples/                   0.14         90.24    827.0          66.0                  633.3
extensions/                31.48        896.41     30.0          10.0                   28.5
frontend/                   1.54         43.65    194.0          25.0                   28.3
internal/                   1.69        111.77     82.0          17.0                   66.1
lib/                        2.01         54.53    248.0          34.0                   27.1
libs/                       1.98        115.02     57.0          28.0                   58.2
modules/                   18.68        190.53     51.0          13.0                   10.2
packages/                   2.76         54.01    678.0         119.0                   19.6
pkg/                        0.05         33.96    190.0          32.0                  633.5
public/                     4.34         18.05     60.0          19.0                    4.2
python/                     5.16        140.53     19.0          24.0                   27.2
samples/                    1.00        397.85     85.0          29.0                  399.0
sdk/                       44.32        167.06     26.0          16.0                    3.8
services/                   0.30        580.44     25.0           1.0                 1910.0
src/                        0.25         40.99   2725.0         272.0                  163.6
static/                     0.18        256.25    240.0           4.0                 1426.9
transpiler/                 0.00           NaN   1473.0           NaN                    NaN
ui/                         9.91        250.48     10.0           8.0                   25.3
web/                        0.49         17.24     63.0          20.0                   35.5
website/                    0.75         78.26     20.0           9.0                  103.7

### 2.2 Test Verification Inclusion (Maturity Baseline)

The verification axis of our deterministic multi-dimensional matching framework captures code maturity and verification rigor.
Pull requests that introduce or modify automated test suites undergo more stringent verification,
potentially altering review turnaround time and velocity dynamics.

### Objectives in this Section:
1. Extract test status: `has_tests` boolean via centralized pipeline utilities (`test_count > 0` for AIDev v5 and `parse_mosaic_files_array` for MOSAIC-3M).
2. Evaluate balance (value counts and shares) to verify that `has_tests` is well-conditioned for regression modeling and exact strata matching.
3. Compare median `merge_latency` across `has_tests == True` vs. `has_tests == False`.

In [6]:
print("=== 2.2 EDA: Test Verification Inclusion & Merge Latency ===")

# 1. AIDev v5: Assign has_tests boolean from test_count
df_aidev_pr_filtered['has_tests'] = df_aidev_pr_filtered['test_count'] > 0

# 2. MOSAIC-3M: has_tests is already extracted from the files array via parse_mosaic_files_array

# 3. Value counts and percentages
aidev_counts = df_aidev_pr_filtered["has_tests"].value_counts()
aidev_shares = (df_aidev_pr_filtered["has_tests"].value_counts(normalize=True) * 100).round(2)
print("\n--- AIDev v5 Test Inclusion (has_tests) Balance ---")
print(aidev_counts.map('{:,}'.format) + " (" + aidev_shares.map('{:.2f}%'.format) + ")")

mosaic_counts = df_mosaic_pr_filtered["has_tests"].value_counts()
mosaic_shares = (df_mosaic_pr_filtered["has_tests"].value_counts(normalize=True) * 100).round(2)
print("\n--- MOSAIC-3M Test Inclusion (has_tests) Balance ---")
print(mosaic_counts.map('{:,}'.format) + " (" + mosaic_shares.map('{:.2f}%'.format) + ")")

# 4. Summary Table comparing median latency across has_tests
test_latency_summary = pd.DataFrame({
    "AIDev PR Count": aidev_counts,
    "AIDev Share (%)": aidev_shares,
    "AIDev Median Latency (h)": df_aidev_pr_filtered.groupby("has_tests")["merge_latency"].median().round(2),
    "MOSAIC PR Count": mosaic_counts,
    "MOSAIC Share (%)": mosaic_shares,
    "MOSAIC Median Latency (h)": df_mosaic_pr_filtered.groupby("has_tests")["merge_latency"].median().round(2)
})
print("\n--- Median Merge Latency (Hours) by Test Inclusion ---")
display(test_latency_summary)


=== 2.2 EDA: Test Verification Inclusion & Merge Latency ===

--- AIDev v5 Test Inclusion (has_tests) Balance ---
has_tests
False    7,818 (53.01%)
True     6,929 (46.99%)
dtype: object

--- MOSAIC-3M Test Inclusion (has_tests) Balance ---
has_tests
False    2,319 (63.57%)
True     1,329 (36.43%)
dtype: object

--- Median Merge Latency (Hours) by Test Inclusion ---


,AIDev PR Count,AIDev Share (%),AIDev Median Latency (h),MOSAIC PR Count,MOSAIC Share (%),MOSAIC Median Latency (h)
has_tests,,,,,,
False,7818,53.01,0.10,2319,63.57,2.71
True,6929,46.99,0.05,1329,36.43,10.35


### 2.3 Unmatched Infrastructure Category Latency Differentials

Infrastructure modifications are not homogeneous: updating a CI workflow, provisioning cloud resources via IaC,
configuring a container environment, or running automation scripts carry vastly different operational risk profiles.

### Objectives in this Section:
1. Extract `infra_category` for every file from `file_classes` (`CI_CD`, `Containers`, `Automation`, `Provisioning`).
2. Map categories to the PR level and filter strictly for PRs in the dual-scope `App_and_Infra` cohort.
3. Group dual-scope PRs by specific infrastructure category to identify latency differentials and integration bottlenecks.

In [7]:
print("=== 2.3 EDA: Infrastructure Category Latency Differentials ===")

# 1. AIDev v5: Extract infra_category for files in App_and_Infra scope
df_pr_infra = df_commit_dedup[df_commit_dedup["is_infra"] & df_commit_dedup["infra_category"].notna()].copy()
df_pr_infra = df_pr_infra.merge(pr_scope[["pr_id", "scope"]], on="pr_id")
df_pr_infra = df_pr_infra.merge(df_aidev_pr_filtered[["id", "merge_latency"]], left_on="pr_id", right_on="id")

# Filter strictly for App_and_Infra scope
df_app_infra = df_pr_infra[df_pr_infra["scope"] == "App_and_Infra"].copy()

# Deduplicate to (pr_id, infra_category) so each PR counts once per category touched
pr_infra_unique = df_app_infra.drop_duplicates(subset=["pr_id", "infra_category"])

aidev_infra_table = pr_infra_unique.groupby("infra_category")["merge_latency"].agg(
    PR_Count="count",
    Median_Latency_Hours="median",
    Mean_Latency_Hours="mean"
).reset_index()
aidev_infra_table["Share (%)"] = (aidev_infra_table["PR_Count"] / aidev_infra_table["PR_Count"].sum() * 100).round(2)

# Sort by prevalence
order = ["CI_CD", "Automation", "Containers", "Provisioning"]
aidev_infra_table["cat_order"] = aidev_infra_table["infra_category"].map({c: i for i, c in enumerate(order)})
aidev_infra_table = aidev_infra_table.sort_values("cat_order").drop(columns=["cat_order"])

print("\n--- AIDev v5 App_and_Infra Infrastructure Category Breakdown ---")
aidev_disp = aidev_infra_table.copy()
aidev_disp["PR_Count"] = aidev_disp["PR_Count"].map('{:,}'.format)
aidev_disp["Median_Latency_Hours"] = aidev_disp["Median_Latency_Hours"].round(2)
aidev_disp["Mean_Latency_Hours"] = aidev_disp["Mean_Latency_Hours"].round(2)
display(aidev_disp[["infra_category", "PR_Count", "Share (%)", "Median_Latency_Hours", "Mean_Latency_Hours"]])

# 2. MOSAIC-3M: Extract infra_category for files in App_and_Infra scope
mosaic_pr_cat_rows = []
for idx, row in df_mosaic_pr_filtered.iterrows():
    files_val = row["files"]
    if not isinstance(files_val, (list, np.ndarray)):
        continue
    app_c = 0
    infra_cats = set()
    for f_item in files_val:
        if isinstance(f_item, dict) and "path" in f_item and f_item["path"]:
            file_changes = (f_item.get("additions") or 0) + (f_item.get("deletions") or 0)
            res = classify_file_domain(f_item["path"], f_item.get("change_type"), file_changes)
            if res["is_app"]:
                app_c += 1
            if res["is_infra"] and res["infra_category"]:
                infra_cats.add(res["infra_category"])
    if app_c > 0 and len(infra_cats) > 0:
        for cat in infra_cats:
            mosaic_pr_cat_rows.append({
                "id": row["id"],
                "infra_category": cat,
                "merge_latency": row["merge_latency"]
            })

df_mosaic_infra = pd.DataFrame(mosaic_pr_cat_rows)
mosaic_infra_table = df_mosaic_infra.groupby("infra_category")["merge_latency"].agg(
    PR_Count="count",
    Median_Latency_Hours="median",
    Mean_Latency_Hours="mean"
).reset_index()
mosaic_infra_table["Share (%)"] = (mosaic_infra_table["PR_Count"] / mosaic_infra_table["PR_Count"].sum() * 100).round(2)
mosaic_infra_table["cat_order"] = mosaic_infra_table["infra_category"].map({c: i for i, c in enumerate(order)})
mosaic_infra_table = mosaic_infra_table.sort_values("cat_order").drop(columns=["cat_order"])

print("\n--- MOSAIC-3M App_and_Infra Infrastructure Category Breakdown ---")
mosaic_disp = mosaic_infra_table.copy()
mosaic_disp["PR_Count"] = mosaic_disp["PR_Count"].map('{:,}'.format)
mosaic_disp["Median_Latency_Hours"] = mosaic_disp["Median_Latency_Hours"].round(2)
mosaic_disp["Mean_Latency_Hours"] = mosaic_disp["Mean_Latency_Hours"].round(2)
display(mosaic_disp[["infra_category", "PR_Count", "Share (%)", "Median_Latency_Hours", "Mean_Latency_Hours"]])

# 3. Combined comparative table
comp_infra = pd.DataFrame({
    "AIDev PRs": aidev_infra_table.set_index("infra_category")["PR_Count"],
    "AIDev Share (%)": aidev_infra_table.set_index("infra_category")["Share (%)"],
    "AIDev Median Latency (h)": aidev_infra_table.set_index("infra_category")["Median_Latency_Hours"].round(2),
    "MOSAIC PRs": mosaic_infra_table.set_index("infra_category")["PR_Count"],
    "MOSAIC Share (%)": mosaic_infra_table.set_index("infra_category")["Share (%)"],
    "MOSAIC Median Latency (h)": mosaic_infra_table.set_index("infra_category")["Median_Latency_Hours"].round(2)
})
print("\n--- Cross-Dataset Comparison: Median Merge Latency by Infrastructure Category ---")
display(comp_infra)

=== 2.3 EDA: Infrastructure Category Latency Differentials ===

--- AIDev v5 App_and_Infra Infrastructure Category Breakdown ---


,infra_category,PR_Count,Share (%),Median_Latency_Hours,Mean_Latency_Hours
1,CI_CD,755,61.13,39.29,165.15
0,Automation,199,16.11,78.26,253.92
2,Containers,184,14.90,32.04,189.78
3,Provisioning,97,7.85,32.89,164.09



--- MOSAIC-3M App_and_Infra Infrastructure Category Breakdown ---


,infra_category,PR_Count,Share (%),Median_Latency_Hours,Mean_Latency_Hours
1,CI_CD,144,62.07,4.15,53.17
0,Automation,27,11.64,26.21,86.23
2,Containers,43,18.53,4.10,81.26
3,Provisioning,18,7.76,24.51,94.22



--- Cross-Dataset Comparison: Median Merge Latency by Infrastructure Category ---


,AIDev PRs,AIDev Share (%),AIDev Median Latency (h),MOSAIC PRs,MOSAIC Share (%),MOSAIC Median Latency (h)
infra_category,,,,,,
CI_CD,755,61.13,39.29,144,62.07,4.15
Automation,199,16.11,78.26,27,11.64,26.21
Containers,184,14.90,32.04,43,18.53,4.10
Provisioning,97,7.85,32.89,18,7.76,24.51


## Section 3: Matching Strata Preparation & Common Support Validation

### 3.1 Architectural Archetype Mapping & Pre-Match Overlap

To prepare our cohorts for **Exact Multidimensional Strata Matching with 1D Nearest-Neighbor Caliper Pairing**, we perform structural standardization, architectural classification, cohort balance checks, and volume matching preparation:
1. **1-to-1 PR Mapping & Architectural Archetypes (`scope` & `architectural_archetype`)**:
   - **AIDev v5**: Map `scope` from `pr_scope` and compute `primary_root_dir` using application files only (`is_app == True` in `df_commit_dedup`) per PR to prevent treatment infrastructure files from confounding the architectural location. If multiple root directories tie for the highest file count, assign `"Cross-Domain"` as a deterministic tie-breaker.
   - **MOSAIC-3M**: Iterate over the nested `files` column using `classify_file_domain` to determine `scope` (`App_Only`, `App_and_Infra`, `Infra_Only`, `Other`) and `parse_file_metadata` to find the most frequent `primary_root_dir`. If multiple root directories share the maximum count, assign `"Cross-Domain"`.
   - **Retain Complete Empirical Cohorts**: Rather than dropping non-standard directory PRs (which previously caused severe survivorship bias and dropped ~79% of valid PRs), we retain ALL PRs passing Tier 1–4 classification.
   - **Architectural Archetypes**: Map `primary_root_dir` into expanded structural archetypes:
     - `Core/Library`: `src/`, `lib/`, `libs/`, `core/`, `packages/`, `pkg/`, `crates/`, `modules/`, `sdk/`, `internal/`, `compiler/`, `transpiler/`, `compile/`, `cmd/`, `source/`, `sources/`, `Sources/`
     - `Frontend/App`: `app/`, `apps/`, `frontend/`, `web/`, `ui/`, `client/`, `public/`, `static/`
     - `Backend/API`: `backend/`, `api/`, `server/`, `services/`
     - `Examples/Samples`: `examples/`, `samples/`, `demo/`
     - `Vendor/Dependencies`: `vendor/`, `third_party/`
     - `Root (Top-Level)`: `(root)`
     - `Cross-Domain/Full-Stack`: `Cross-Domain` (tied root directories preventing cross-matching of full-stack with single-domain PRs)
     - `Root/Other`: `(none)`, and any other unmapped directory.
2. **Agent vs. Scope Cohort Overlap (EDA)**:
   - Cross-tabulate `agent` vs. `scope` (AIDev v5) and `agent_source` vs. `scope` (MOSAIC-3M) across the complete retained population.
3. **Common Support Validation (EDA)**:
   - Construct **Common Support Matrices** across `architectural_archetype` $\times$ `has_tests` strata.
   - Verify that each exact-match bucket possesses high density and valid control (`App_Only`) and treatment (`App_and_Infra`) units (`has_support = True`).
4. **Exact Strata Matching Specification (Post-Treatment Bias Correction)**:
   - **Exact Matching Strata**: Exactly partitioned across 4 categorical dimensions (`agent`, `architectural_archetype`, `has_tests`, `ecosystem`) to eliminate agent-specific, structural, maturity, and primary language confounding without relying on a logistic propensity score.
   - **Nearest-Neighbor Distance**: Minimizes 1D standardized Euclidean distance on log application file volume ($z_{\log(\text{app\_count})}$) with a strict $\epsilon \le 0.28$ standard deviation caliper.
   - **Primary Outcome Variable**: Evaluates integration velocity penalty on `merge_latency` (hours).

In [8]:
print("=== 3.1 Exact Strata Preparation & Common Support Validation ===") 

# --------------------------------------------------------------------------
# 1. Assign scope, primary_root_dir (1-to-1 PR Mapping), and architectural_archetype
# --------------------------------------------------------------------------
# A. AIDev v5: Merge scope from pr_scope and compute primary_root_dir from df_commit_dedup
if "pr_id" in df_aidev_pr_filtered.columns:
    df_aidev_pr_filtered.drop(columns=["pr_id"], inplace=True)
df_aidev_pr_filtered = df_aidev_pr_filtered.merge(pr_scope[["pr_id", "scope"]], left_on="id", right_on="pr_id", how="left")
if "pr_id" in df_aidev_pr_filtered.columns:
    df_aidev_pr_filtered.drop(columns=["pr_id"], inplace=True)
df_aidev_pr_filtered["scope"] = df_aidev_pr_filtered["scope"].fillna("Other")

# AIDev Tie-Breaker: Isolate the Architectural Archetype using application files only (is_app == True)
# to prevent infrastructure files (treatment) from forcing ties or altering the architectural archetype.
df_app_commits = df_commit_dedup[df_commit_dedup["is_app"] == True].copy()
root_counts = df_app_commits.groupby(["pr_id", "root_dir"]).size().reset_index(name="file_count")
max_counts = root_counts.groupby("pr_id")["file_count"].transform("max")
top_roots = root_counts[root_counts["file_count"] == max_counts]
tie_counts = top_roots.groupby("pr_id").size()
tied_pr_ids = set(tie_counts[tie_counts > 1].index)

primary_roots = top_roots.drop_duplicates(subset=["pr_id"], keep="first").copy()
primary_roots.loc[primary_roots["pr_id"].isin(tied_pr_ids), "root_dir"] = "Cross-Domain"
primary_roots.rename(columns={"root_dir": "primary_root_dir"}, inplace=True)
df_aidev_pr_filtered = df_aidev_pr_filtered.merge(primary_roots[["pr_id", "primary_root_dir"]], left_on="id", right_on="pr_id", how="left")
if "pr_id" in df_aidev_pr_filtered.columns:
    df_aidev_pr_filtered.drop(columns=["pr_id"], inplace=True)
df_aidev_pr_filtered["primary_root_dir"] = df_aidev_pr_filtered["primary_root_dir"].fillna("(none)")

# B. MOSAIC-3M: Extract scope and primary_root_dir from files array (vectorized parser)
mosaic_meta = df_mosaic_pr_filtered["files"].apply(parse_mosaic_files_array)
df_mosaic_pr_filtered["scope"] = mosaic_meta.apply(lambda x: assign_pr_scope(x['app_count'], x['infra_count']), axis=1)
df_mosaic_pr_filtered["primary_root_dir"] = mosaic_meta["primary_root_dir"]

# C. Map primary_root_dir into Architectural Archetypes (imported from src.pipeline_utils)
df_aidev_pr_filtered["architectural_archetype"] = df_aidev_pr_filtered["primary_root_dir"].map(map_architectural_archetype)
df_mosaic_pr_filtered["architectural_archetype"] = df_mosaic_pr_filtered["primary_root_dir"].map(map_architectural_archetype)

print(f"AIDev PRs Retained (Complete Cohort): {len(df_aidev_pr_filtered):,}")
print(f"MOSAIC-3M PRs Retained (Complete Cohort): {len(df_mosaic_pr_filtered):,}")

print("\n--- AIDev v5 Scope Distribution ---")
print(df_aidev_pr_filtered["scope"].value_counts().to_string())
print("\n--- MOSAIC-3M Scope Distribution ---")
print(df_mosaic_pr_filtered["scope"].value_counts().to_string())

print("\n--- AIDev v5 Architectural Archetype Distribution ---")
print(df_aidev_pr_filtered["architectural_archetype"].value_counts().to_string())
print("\n--- MOSAIC-3M Architectural Archetype Distribution ---")
print(df_mosaic_pr_filtered["architectural_archetype"].value_counts().to_string())

# --------------------------------------------------------------------------
# 2. Agent vs. Scope Cohort Overlap (EDA)
# --------------------------------------------------------------------------
print("\n--- AIDev v5: Agent vs. Scope Cohort Overlap (Crosstab) ---")
aidev_crosstab = pd.crosstab(df_aidev_pr_filtered["agent"], df_aidev_pr_filtered["scope"], margins=True)
print(aidev_crosstab.to_string())

print("\n--- MOSAIC-3M: Agent vs. Scope Cohort Overlap (Crosstab) ---")
mosaic_crosstab = pd.crosstab(df_mosaic_pr_filtered["agent_source"], df_mosaic_pr_filtered["scope"], margins=True)
print(mosaic_crosstab.to_string())

# --------------------------------------------------------------------------
# 3. Common Support Validation (EDA)
# --------------------------------------------------------------------------
aidev_support = df_aidev_pr_filtered.groupby(["architectural_archetype", "has_tests", "scope"]).size().unstack(level="scope").fillna(0).astype(int)
for col in ["App_Only", "App_and_Infra"]:
    if col not in aidev_support.columns:
        aidev_support[col] = 0
aidev_support["has_support"] = (aidev_support["App_Only"] > 0) & (aidev_support["App_and_Infra"] > 0)

print("\n--- AIDev v5: Common Support Matrix (architectural_archetype x has_tests) ---")
print(aidev_support.to_string())

mosaic_support = df_mosaic_pr_filtered.groupby(["architectural_archetype", "has_tests", "scope"]).size().unstack(level="scope").fillna(0).astype(int)
for col in ["App_Only", "App_and_Infra"]:
    if col not in mosaic_support.columns:
        mosaic_support[col] = 0
mosaic_support["has_support"] = (mosaic_support["App_Only"] > 0) & (mosaic_support["App_and_Infra"] > 0)

print("\n--- MOSAIC-3M: Common Support Matrix (architectural_archetype x has_tests) ---")
print(mosaic_support.to_string())


=== 3.1 Exact Strata Preparation & Common Support Validation ===


AIDev PRs Retained (Complete Cohort): 14,747
MOSAIC-3M PRs Retained (Complete Cohort): 3,648

--- AIDev v5 Scope Distribution ---
scope
App_Only         13768
App_and_Infra      979

--- MOSAIC-3M Scope Distribution ---
scope
App_Only         3435
App_and_Infra     213

--- AIDev v5 Architectural Archetype Distribution ---
architectural_archetype
Core/Library               7900
Root/Other                 3572
Frontend/App               1117
Examples/Samples            704
Root (Top-Level)            667
Cross-Domain/Full-Stack     649
Backend/API                 130
Vendor/Dependencies           8

--- MOSAIC-3M Architectural Archetype Distribution ---
architectural_archetype
Core/Library               1468
Root/Other                 1075
Frontend/App                639
Cross-Domain/Full-Stack     185
Root (Top-Level)            168
Backend/API                  65
Examples/Samples             43
Vendor/Dependencies           5

--- AIDev v5: Agent vs. Scope Cohort Overlap (Crosstab) --

### 3.2 Diagnostic Audit: 4D Strata Cell Occupancy & Orphan Treatment Analysis

To evaluate common support under our 4D exact matching strata (`agent` $\times$ `architectural_archetype` $\times$ `has_tests` $\times$ `ecosystem`), we audit cell density, multidimensional occupancy, and orphan treatment rates across both datasets before caliper matching execution.

In [9]:
# ==============================================================================
# SECTION 6.1: DIAGNOSTIC EDA ON STRATA COMPOSITION & CELL OCCUPANCY
# ==============================================================================

from src.pipeline_utils import map_ecosystem

print("=" * 80)
print("=== 3.2 DIAGNOSTIC EDA: 4D STRATA OCCUPANCY & ORPHAN TREATMENT AUDIT ===")
print("=" * 80)

# Set clean tabular display
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# 0. Defensive Environment Setup
# Map ecosystem defensively onto the filtered base PR tables
if "ecosystem" not in df_aidev_pr_filtered.columns and "primary_language" in df_aidev_pr_filtered.columns:
    df_aidev_pr_filtered["ecosystem"] = df_aidev_pr_filtered["primary_language"].apply(map_ecosystem)

if "ecosystem" not in df_mosaic_pr_filtered.columns and "primary_language" in df_mosaic_pr_filtered.columns:
    df_mosaic_pr_filtered["ecosystem"] = df_mosaic_pr_filtered["primary_language"].apply(map_ecosystem)

# Instantiate / Refresh fresh agentic slice for MOSAIC-3M
df_mosaic_agentic = df_mosaic_pr_filtered[df_mosaic_pr_filtered["agent_source"] != "Human"].copy()

# Helper: Clean display of margins crosstab keeping 'All' pinned at the bottom
def display_sorted_crosstab(ct, sort_col="All"):
    cat_rows = [r for r in ct.index if r != "All"]
    ct_sorted = ct.loc[cat_rows].sort_values(by=sort_col, ascending=False)
    if "All" in ct.index:
        ct_sorted = pd.concat([ct_sorted, ct.loc[["All"]]])
    print(ct_sorted.to_string())

# ------------------------------------------------------------------------------
# 1. ARCHITECTURAL ARCHETYPE CENSUS (TREATMENT VS. CONTROL)
# ------------------------------------------------------------------------------
print("\n--- 1. Architectural Archetype Scope Breakdown ---")

def audit_archetype_census(df, dataset_name):
    ct = pd.crosstab(df["architectural_archetype"], df["scope"], margins=True)
    if "App_and_Infra" in ct.columns:
        total_treat = ct.loc["All", "App_and_Infra"]
        ct["Treat_Prevalence_in_Archetype (%)"] = (ct["App_and_Infra"] / ct["All"] * 100).round(2)
        ct["Share_of_Total_Treatment (%)"] = (ct["App_and_Infra"] / total_treat * 100).round(2) if total_treat > 0 else 0.0
    print(f"\n[{dataset_name}] Archetype Distribution:")
    display_sorted_crosstab(ct, sort_col="App_and_Infra" if "App_and_Infra" in ct.columns else "All")

audit_archetype_census(df_aidev_pr_filtered, "AIDev v5")
audit_archetype_census(df_mosaic_agentic, "MOSAIC-3M (Agentic Only)")

# ------------------------------------------------------------------------------
# 2. UNPACKING THE 'ROOT/OTHER' JUNK DRAWER
# ------------------------------------------------------------------------------
print("\n--- 2. What Is Inside 'Root/Other'? (Top 20 Directories) ---")

def audit_root_other(df, dataset_name):
    other_prs = df[df["architectural_archetype"] == "Root/Other"]
    top_dirs = other_prs.groupby("primary_root_dir", dropna=False).agg(
        Total_PRs=("scope", "count"),
        Treatment_PRs=("scope", lambda x: (x == "App_and_Infra").sum()),
        Control_PRs=("scope", lambda x: (x == "App_Only").sum())
    ).sort_values(by="Total_PRs", ascending=False).head(20)
    top_dirs["Treat_Share (%)"] = (top_dirs["Treatment_PRs"] / top_dirs["Total_PRs"] * 100).round(2)
    print(f"\n[{dataset_name}] Top 20 Directories in 'Root/Other' (Total PRs: {len(other_prs):,}):")
    print(top_dirs.to_string())

audit_root_other(df_aidev_pr_filtered, "AIDev v5")
audit_root_other(df_mosaic_agentic, "MOSAIC-3M (Agentic Only)")

# ------------------------------------------------------------------------------
# 3. ECOSYSTEM & PRIMARY LANGUAGE CENSUS
# ------------------------------------------------------------------------------
print("\n--- 3. Language Ecosystem Breakdown ---")

def audit_ecosystem_census(df, dataset_name):
    ct = pd.crosstab(df["ecosystem"], df["scope"], margins=True)
    if "App_and_Infra" in ct.columns:
        total_treat = ct.loc["All", "App_and_Infra"]
        ct["Treat_Share_in_Ecosystem (%)"] = (ct["App_and_Infra"] / ct["All"] * 100).round(2)
        ct["Share_of_Total_Treatment (%)"] = (ct["App_and_Infra"] / total_treat * 100).round(2) if total_treat > 0 else 0.0
    print(f"\n[{dataset_name}] Ecosystem Distribution:")
    display_sorted_crosstab(ct, sort_col="App_and_Infra" if "App_and_Infra" in ct.columns else "All")

audit_ecosystem_census(df_aidev_pr_filtered, "AIDev v5")
audit_ecosystem_census(df_mosaic_agentic, "MOSAIC-3M (Agentic Only)")

print("\n[MOSAIC-3M] Raw Unmapped Primary Languages for Dual-Scope PRs:")
mosaic_treat_raw_langs = df_mosaic_agentic[df_mosaic_agentic["scope"] == "App_and_Infra"]["primary_language"].value_counts()
print(mosaic_treat_raw_langs.to_frame(name="MOSAIC_Treatment_PR_Count").to_string())

# ------------------------------------------------------------------------------
# 4. CELL OCCUPANCY & ORPHAN TREATMENT AUDIT (2D AND FULL 3D STRATA)
# ------------------------------------------------------------------------------
print("\n--- 4. Cell Density & Orphan Treatment Audit ---")

def audit_strata_occupancy(df, dataset_name):
    treat = df[df["scope"] == "App_and_Infra"]
    ctrl = df[df["scope"] == "App_Only"]
    
    # 4A. 2D Evaluation: Archetype x Ecosystem
    all_archetypes = sorted(list(set(treat["architectural_archetype"]).union(set(ctrl["architectural_archetype"]))))
    all_ecosystems = sorted(list(set(treat["ecosystem"]).union(set(ctrl["ecosystem"]))))
    
    treat_2d = pd.crosstab(treat["architectural_archetype"], treat["ecosystem"]).reindex(
        index=all_archetypes, columns=all_ecosystems, fill_value=0
    )
    ctrl_2d = pd.crosstab(ctrl["architectural_archetype"], ctrl["ecosystem"]).reindex(
        index=all_archetypes, columns=all_ecosystems, fill_value=0
    )
    
    orphans_2d = (treat_2d > 0) & (ctrl_2d == 0)
    orphan_prs_2d = treat_2d[orphans_2d].sum().sum()
    
    print(f"\n=======================================================")
    print(f"[{dataset_name}] 2D & 3D STRATA OCCUPANCY AUDIT")
    print(f"=======================================================")
    print(f"Total Treatment PRs: {len(treat):,}")
    print(f"Treatment PRs in 2D cells with ZERO control support: {orphan_prs_2d:,} ({orphan_prs_2d / len(treat) * 100:.2f}%)")
    
    print(f"\n[{dataset_name}] 2D Treatment PR Count Grid (Archetype x Ecosystem):")
    print(treat_2d.to_string())
    print(f"\n[{dataset_name}] 2D Available Control Pool Grid (Archetype x Ecosystem):")
    print(ctrl_2d.to_string())
    
    # 4B. Full 3D Strata Evaluation: Archetype x Ecosystem x Test Status
    treat_3d = treat.groupby(["architectural_archetype", "ecosystem", "has_tests"]).size()
    ctrl_3d = ctrl.groupby(["architectural_archetype", "ecosystem", "has_tests"]).size()
    
    df_3d = pd.DataFrame({"Treatment_PRs": treat_3d, "Control_PRs": ctrl_3d}).fillna(0).astype(int)
    orphans_3d = df_3d[(df_3d["Treatment_PRs"] > 0) & (df_3d["Control_PRs"] == 0)]
    total_orphans_3d = orphans_3d["Treatment_PRs"].sum()
    
    active_cells = len(df_3d[df_3d["Treatment_PRs"] > 0])
    zero_cells = len(orphans_3d)
    
    print(f"\n[{dataset_name}] Full 3D Strata Audit (Archetype x Ecosystem x has_tests):")
    print(f" - Active Treatment Strata Cells       : {active_cells}")
    print(f" - Cells with ZERO Control Support     : {zero_cells} ({(zero_cells/active_cells*100):.1f}% of active cells)")
    print(f" - Total Treatment PRs Dropped by 3D   : {total_orphans_3d:,} ({(total_orphans_3d/len(treat)*100):.2f}% of treatment sample)")
    
    if len(orphans_3d) > 0:
        print(f"\n[{dataset_name}] Top Strata Cells Discarding Treatment PRs:")
        print(orphans_3d.sort_values(by="Treatment_PRs", ascending=False).head(10).to_string())

# Execute on both datasets
audit_strata_occupancy(df_aidev_pr_filtered, "AIDev v5")
audit_strata_occupancy(df_mosaic_agentic, "MOSAIC-3M (Agentic Only)")

=== 3.2 DIAGNOSTIC EDA: 4D STRATA OCCUPANCY & ORPHAN TREATMENT AUDIT ===



--- 1. Architectural Archetype Scope Breakdown ---

[AIDev v5] Archetype Distribution:
scope                    App_Only  App_and_Infra    All  Treat_Prevalence_in_Archetype (%)  Share_of_Total_Treatment (%)
architectural_archetype                                                                                                 
Core/Library                 7509            391   7900                               4.95                         39.94
Root/Other                   3273            299   3572                               8.37                         30.54
Frontend/App                 1008            109   1117                               9.76                         11.13
Root (Top-Level)              576             91    667                              13.64                          9.30
Cross-Domain/Full-Stack       587             62    649                               9.55                          6.33
Examples/Samples              687             17    704          


[AIDev v5] Top 20 Directories in 'Root/Other' (Total PRs: 3,572):
                     Total_PRs  Treatment_PRs  Control_PRs  Treat_Share (%)
primary_root_dir                                                           
alpha_factory_v1/          684             45          639             6.58
runtime/                   279              0          279             0.00
aster/                     177              0          177             0.00
custom_components/         140              0          140             0.00
open-isle-cli/             118              0          118             0.00
functions/                  79              1           78             1.27
R/                          40              1           39             2.50
types/                      38              0           38             0.00
jac/                        36              0           36             0.00
interpreter/                32              0           32             0.00
data_access/         


[AIDev v5] 2D & 3D STRATA OCCUPANCY AUDIT
Total Treatment PRs: 979
Treatment PRs in 2D cells with ZERO control support: 0.0 (0.00%)

[AIDev v5] 2D Treatment PR Count Grid (Archetype x Ecosystem):
ecosystem                .NET  Compiled Systems  Interpreted Backend  JVM  Mobile/Native UI  Node/Frontend  Ops/Scripting  Other/Niche  Python
architectural_archetype                                                                                                                       
Backend/API                 0                 0                    0    0                 0              5              0            0       4
Core/Library               96                94                   10   17                18            126              2            2      26
Cross-Domain/Full-Stack     3                17                    0    0                 2             23              0            0      17
Examples/Samples            2                 7                    3    0               

## Section 4: Exact Multidimensional Strata Matching & Balance Verification

### 4.1 Greedy 1:1 Nearest-Neighbor Caliper Matching Execution

To address Research Question 2 (RQ2), we evaluate the causal integration penalty of infrastructure changes (`App_and_Infra` vs. `App_Only`):
1. **Exact Strata Matching**: Exact matching within `agent` $\times$ `architectural_archetype` $\times$ `has_tests` $\times$ `ecosystem` strata guarantees agent model identity, architectural archetype, test verification rigor, and primary language ecosystem equivalence.
2. **Nearest-Neighbor Distance & Strict Caliper (1D Distance)**: Within each stratum, treatment units are sorted descending by application volume to prioritize matching larger PRs before the control pool is depleted. Each Treatment PR (`App_and_Infra`) is paired with an available Control PR (`App_Only`) minimizing 1D standardized Euclidean distance on application volume:
   $$\text{dist}(i, j) = |z_{\log(\text{app\_count})_i} - z_{\log(\text{app\_count})_j}|$$
   A strict caliper constraint ($\text{dist}(i, j) \le 0.28$ standard deviations) is enforced, dropping treatment PRs without a closely sized control unit. Matched control units are removed from the stratum pool without replacement.

### 4.2 Average Treatment Effect on the Treated (ATT) & Covariate Balance Verification

3. **Average Treatment Effect on the Treated (ATT)**:
   - **Velocity Penalty**: Evaluates paired latency difference on `merge_latency` (hours).
4. **Covariate Balance**: Evaluates elimination of pre-treatment volume disparity on `app_count`, while inspecting post-match distributions of `app_line_churn` and `stars`.

In [10]:
print("=== 4.1 Causal Inference: Greedy 1:1 Matching within Exact Strata ===")

# Execute matching on both cohorts using centralized PSM pipeline (with 1D distance matching on volume)
df_aidev_matched, df_mosaic_matched = generate_matched_cohorts(df_aidev_pr_filtered, df_mosaic_pr_filtered)
df_mosaic_agentic = df_mosaic_pr_filtered[df_mosaic_pr_filtered["agent_source"] != "Human"].copy()

# --------------------------------------------------------------------------
# ATT (Average Treatment Effect on the Treated) & Covariate Balance Evaluation
# --------------------------------------------------------------------------
def report_matching_att_and_balance(df_matched: pd.DataFrame, df_orig: pd.DataFrame, dataset_name: str):
    paired_df = df_matched.pivot(
        index="pair_id", 
        columns="match_role", 
        values=["merge_latency", "app_count", "log_app_count", "app_line_churn", "total_files_recorded", "stars", "log_stars"]
    )
    
    latency_diff = paired_df["merge_latency"]["Treatment"] - paired_df["merge_latency"]["Control"]
    
    treat_med_lat = df_matched[df_matched["match_role"] == "Treatment"]["merge_latency"].median()
    ctrl_med_lat = df_matched[df_matched["match_role"] == "Control"]["merge_latency"].median()
    
    print(f"\n--- ATT (Average Treatment Effect on the Treated): {dataset_name} ---")
    print("Primary Outcome: merge_latency (Hours)")
    print(f"  - Median Paired Latency Difference (Treatment - Control): {latency_diff.median():+.2f} hours")
    print(f"  - Difference of Medians (Treatment: {treat_med_lat:.2f}h vs Control: {ctrl_med_lat:.2f}h): {treat_med_lat - ctrl_med_lat:+.2f} hours")
    print(f"  - Mean Paired Latency Difference: {latency_diff.mean():+.2f} hours")
    
    pre_treat_app = df_orig[df_orig["scope"] == "App_and_Infra"]["app_count"]
    pre_ctrl_app = df_orig[df_orig["scope"] == "App_Only"]["app_count"]
    post_treat_app = df_matched[df_matched["match_role"] == "Treatment"]["app_count"]
    post_ctrl_app = df_matched[df_matched["match_role"] == "Control"]["app_count"]

    pre_treat_churn = df_orig[df_orig["scope"] == "App_and_Infra"]["app_line_churn"]
    pre_ctrl_churn = df_orig[df_orig["scope"] == "App_Only"]["app_line_churn"]
    post_treat_churn = df_matched[df_matched["match_role"] == "Treatment"]["app_line_churn"]
    post_ctrl_churn = df_matched[df_matched["match_role"] == "Control"]["app_line_churn"]

    pre_treat_stars = df_orig[df_orig["scope"] == "App_and_Infra"]["stars"]
    pre_ctrl_stars = df_orig[df_orig["scope"] == "App_Only"]["stars"]
    post_treat_stars = df_matched[df_matched["match_role"] == "Treatment"]["stars"]
    post_ctrl_stars = df_matched[df_matched["match_role"] == "Control"]["stars"]

    balance_summary = pd.DataFrame({
        "Pre-Match Treatment": [
            len(pre_treat_app),
            pre_treat_app.mean(), pre_treat_app.median(),
            pre_treat_churn.mean(), pre_treat_churn.median(),
            pre_treat_stars.mean(), pre_treat_stars.median()
        ],
        "Pre-Match Control": [
            len(pre_ctrl_app),
            pre_ctrl_app.mean(), pre_ctrl_app.median(),
            pre_ctrl_churn.mean(), pre_ctrl_churn.median(),
            pre_ctrl_stars.mean(), pre_ctrl_stars.median()
        ],
        "Post-Match Treatment": [
            len(post_treat_app),
            post_treat_app.mean(), post_treat_app.median(),
            post_treat_churn.mean(), post_treat_churn.median(),
            post_treat_stars.mean(), post_treat_stars.median()
        ],
        "Post-Match Control": [
            len(post_ctrl_app),
            post_ctrl_app.mean(), post_ctrl_app.median(),
            post_ctrl_churn.mean(), post_ctrl_churn.median(),
            post_ctrl_stars.mean(), post_ctrl_stars.median()
        ],
    }, index=["PR Count", "Mean App Files", "Median App Files", "Mean App Line Churn", "Median App Line Churn", "Mean Stars", "Median Stars"])

    print(f"\n--- Covariate Balance: Pre-Match vs. Post-Match (app_count, app_line_churn & stars) ---")
    print(balance_summary.round(2).to_string())

report_matching_att_and_balance(df_aidev_matched, df_aidev_pr_filtered, "AIDev v5")
report_matching_att_and_balance(df_mosaic_matched, df_mosaic_agentic, "MOSAIC-3M (Agentic Only)")

=== 4.1 Causal Inference: Greedy 1:1 Matching within Exact Strata ===

Executing Greedy 1:1 Matching: AIDev v5
Treatment: scope == 'App_and_Infra' | Control: 'App_Only'
Strata: ['agent', 'architectural_archetype', 'has_tests', 'ecosystem'] | Distance: ['z_log_app_count']


Successfully Matched: 687 pairs

Executing Greedy 1:1 Matching: MOSAIC-3M (Agentic Only)
Treatment: scope == 'App_and_Infra' | Control: 'App_Only'
Strata: ['agent_source', 'architectural_archetype', 'has_tests', 'ecosystem'] | Distance: ['z_log_app_count']


Successfully Matched: 80 pairs

--- ATT (Average Treatment Effect on the Treated): AIDev v5 ---
Primary Outcome: merge_latency (Hours)
  - Median Paired Latency Difference (Treatment - Control): +0.84 hours
  - Difference of Medians (Treatment: 16.60h vs Control: 3.47h): +13.13 hours
  - Mean Paired Latency Difference: +59.72 hours

--- Covariate Balance: Pre-Match vs. Post-Match (app_count, app_line_churn & stars) ---
                       Pre-Match Treatment  Pre-Match Control  Post-Match Treatment  Post-Match Control
PR Count                            979.00           13768.00                687.00              687.00
Mean App Files                       47.59               5.20                 22.71               20.78
Median App Files                     10.00               1.00                  6.00                6.00
Mean App Line Churn                5275.50             818.39               2555.76             1942.15
Median App Line Churn               446.00              4

### 4.3 Treatment Dose Audit: Infrastructure Payload Distribution

In RQ2, we matched Treatment (`App_and_Infra`) and Control (`App_Only`) units by holding application file volume constant (`z_log_app_count`). To defend construct validity for peer review, we audit the "treatment dose"—specifically, the distribution of infrastructure file counts within the matched Treatment group across both datasets. This proves whether the treatment represents atomic infrastructure modifications (1-2 files) or extreme repository-wide refactoring.

In [11]:
print("=== 4.3 DIAGNOSTIC: TREATMENT DOSE & INFRASTRUCTURE PAYLOAD AUDIT ===")

# --------------------------------------------------------------------------
# 1. AIDev v5: Extract infra_files for Matched Treatment PRs (N=826)
# --------------------------------------------------------------------------
aidev_treat = df_aidev_matched[df_aidev_matched['match_role'] == 'Treatment'][['id', 'pair_id']].copy()
# Re-link with pr_scope from Step 3 which contains deduplicated infra_files
aidev_treat = aidev_treat.merge(pr_scope[['pr_id', 'infra_files']], left_on='id', right_on='pr_id', how='left')
aidev_treat['infra_files'] = aidev_treat['infra_files'].fillna(0).astype(int)
aidev_infra_series = aidev_treat['infra_files']

# --------------------------------------------------------------------------
# 2. MOSAIC-3M: Extract infra_count for Matched Treatment PRs (N=99)
# --------------------------------------------------------------------------
mosaic_treat = df_mosaic_matched[df_mosaic_matched['match_role'] == 'Treatment'][['id', 'pair_id']].copy()
# Defensively ensure files column is present from df_mosaic_pr_filtered
if 'files' in df_mosaic_matched.columns:
    mosaic_treat['files'] = df_mosaic_matched.loc[mosaic_treat.index, 'files']
else:
    mosaic_treat = mosaic_treat.merge(df_mosaic_pr_filtered[['id', 'files']], on='id', how='left')

# Use parse_mosaic_files_array for 100% deduplication and taxonomy parity
mosaic_infra_series = mosaic_treat['files'].apply(
    lambda f: parse_mosaic_files_array(f)['infra_count']
).astype(int)

# --------------------------------------------------------------------------
# 3. Construct Comparative Treatment Dose Summary Table
# --------------------------------------------------------------------------
def summarize_infra_dose(series: pd.Series, dataset_name: str) -> pd.Series:
    n = len(series)
    c1 = int((series == 1).sum())
    c2_3 = int(((series >= 2) & (series <= 3)).sum())
    c4_plus = int((series >= 4).sum())
    
    return pd.Series({
        "Matched Treatment PRs": f"{n:,}",
        "Mean Infra Files": f"{series.mean():.2f}",
        "Std Dev": f"{series.std():.2f}",
        "Min": f"{int(series.min())}",
        "25th Percentile (Q1)": f"{series.quantile(0.25):.1f}",
        "Median (50th)": f"{series.median():.1f}",
        "75th Percentile (Q3)": f"{series.quantile(0.75):.1f}",
        "90th Percentile": f"{series.quantile(0.90):.1f}",
        "Max": f"{int(series.max())}",
        "Exactly 1 Infra File": f"{c1:,} ({(c1 / n * 100):.2f}%)",
        "2-3 Infra Files": f"{c2_3:,} ({(c2_3 / n * 100):.2f}%)",
        "4+ Infra Files": f"{c4_plus:,} ({(c4_plus / n * 100):.2f}%)"
    }, name=dataset_name)

df_dose_summary = pd.DataFrame([
    summarize_infra_dose(aidev_infra_series, "AIDev v5 (Matched Treatment)"),
    summarize_infra_dose(mosaic_infra_series, "MOSAIC-3M (Matched Treatment)")
]).T

print("\n--- Infrastructure Treatment Dose Distribution (Matched Treatment Units) ---")
display(df_dose_summary)

=== 4.3 DIAGNOSTIC: TREATMENT DOSE & INFRASTRUCTURE PAYLOAD AUDIT ===

--- Infrastructure Treatment Dose Distribution (Matched Treatment Units) ---


,AIDev v5 (Matched Treatment),MOSAIC-3M (Matched Treatment)
Matched Treatment PRs,687,80
Mean Infra Files,3.11,1.80
Std Dev,6.71,1.46
Min,1,1
25th Percentile (Q1),1.0,1.0
Median (50th),1.0,1.0
75th Percentile (Q3),3.0,2.0
90th Percentile,6.0,3.0
Max,117,9
Exactly 1 Infra File,374 (54.44%),49 (61.25%)


### 4.4 Application Source Code vs. Configuration Churn Decomposition

To determine whether the elevated application line churn observed in the Treatment group (`App_and_Infra`) is driven by genuine application source code modifications (`sub_type == 'App'`) or auto-generated configuration/package lockfiles (`sub_type == 'App_Config'`), we perform a fine-grained file-level audit across the matched cohorts:
1. **AIDev v5**: Merge `df_aidev_matched` with `df_commit_dedup` and apply `classify_file_domain` to isolate `'App'` and `'App_Config'` files.
2. **MOSAIC-3M**: Merge `df_mosaic_matched` with `df_mosaic_pr_filtered[['id', 'files']]` and evaluate file-level churn for `'App'` and `'App_Config'` files.
3. **Aggregation**: Group by `match_role` (`Treatment` vs. `Control`) and `sub_type` to calculate the Total Churn and Median Churn per file.

In [12]:
print("=== 4.4 DIAGNOSTIC: APPLICATION SOURCE CODE VS. CONFIGURATION CHURN BREAKDOWN ===")

# --------------------------------------------------------------------------
# 1. AIDev v5: File-Level Churn Decomposition (App vs. App_Config)
# --------------------------------------------------------------------------
aidev_merged = df_aidev_matched[['id', 'match_role']].merge(
    df_commit_dedup[['pr_id', 'filename', 'status', 'changes']],
    left_on='id',
    right_on='pr_id',
    how='inner'
)

# Extract sub_type for each file using pipeline classifier
aidev_merged['sub_type'] = [
    classify_file_domain(fn, st, ch).get('sub_type')
    for fn, st, ch in zip(aidev_merged['filename'], aidev_merged['status'], aidev_merged['changes'])
]

# Filter strictly for App and App_Config
aidev_app_files = aidev_merged[aidev_merged['sub_type'].isin(['App', 'App_Config'])].copy()

# Group by match_role and sub_type
aidev_churn_summary = aidev_app_files.groupby(['match_role', 'sub_type']).agg(
    Total_Churn=('changes', 'sum'),
    Median_Churn=('changes', 'median')
).reset_index().rename(columns={
    'match_role': 'Match Role',
    'sub_type': 'Sub Type',
    'Total_Churn': 'Total Churn',
    'Median_Churn': 'Median Churn'
})

print("\n--- AIDev v5: Application Churn Breakdown (Matched Cohort) ---")
display(aidev_churn_summary)

# --------------------------------------------------------------------------
# 2. MOSAIC-3M: File-Level Churn Decomposition (App vs. App_Config)
# --------------------------------------------------------------------------
mosaic_merged = df_mosaic_matched[['id', 'match_role']].merge(
    df_mosaic_pr_filtered[['id', 'files']],
    on='id',
    how='inner'
)

mosaic_records = []
for _, row in mosaic_merged.iterrows():
    m_role = row['match_role']
    files_val = row['files']
    if isinstance(files_val, (list, np.ndarray)):
        for f_item in files_val:
            if isinstance(f_item, dict):
                path = f_item.get('path') or ''
                ch_type = f_item.get('change_type')
                additions = f_item.get('additions') or 0
                deletions = f_item.get('deletions') or 0
            else:
                path = str(f_item)
                ch_type = None
                additions, deletions = 0, 0
            churn = additions + deletions
            res = classify_file_domain(path, ch_type, churn)
            st = res.get('sub_type')
            if st in ['App', 'App_Config']:
                mosaic_records.append({
                    'Match Role': m_role,
                    'Sub Type': st,
                    'churn': churn
                })

df_mosaic_files = pd.DataFrame(mosaic_records)

mosaic_churn_summary = df_mosaic_files.groupby(['Match Role', 'Sub Type']).agg(
    Total_Churn=('churn', 'sum'),
    Median_Churn=('churn', 'median')
).reset_index().rename(columns={
    'Total_Churn': 'Total Churn',
    'Median_Churn': 'Median Churn'
})

print("\n--- MOSAIC-3M (Agentic Only): Application Churn Breakdown (Matched Cohort) ---")
display(mosaic_churn_summary)


=== 4.4 DIAGNOSTIC: APPLICATION SOURCE CODE VS. CONFIGURATION CHURN BREAKDOWN ===



--- AIDev v5: Application Churn Breakdown (Matched Cohort) ---


,Match Role,Sub Type,Total Churn,Median Churn
0,Control,App,1136523.0,14.0
1,Control,App_Config,197734.0,6.0
2,Treatment,App,940729.0,14.0
3,Treatment,App_Config,815078.0,7.0



--- MOSAIC-3M (Agentic Only): Application Churn Breakdown (Matched Cohort) ---


,Match Role,Sub Type,Total Churn,Median Churn
0,Control,App,86695.0,26.0
1,Control,App_Config,9009.0,5.0
2,Treatment,App,53852.0,30.0
3,Treatment,App_Config,80316.0,4.0


## Section 5: Multidimensional Velocity Friction Analysis

To move beyond aggregate summary statistics, we analyze the integration penalty across three comprehensive dimensions:

### 5.1 Dimension 1: Macro Latency Inflation & Wilcoxon Paired Significance
- **Non-Parametric Paired Testing**: Wilcoxon signed-rank test and Cliff's Delta effect size on paired latency differences ($\Delta = \text{Latency}_{\text{Treat}} - \text{Latency}_{\text{Control}}$).
- *Statistical Note: We apply the Pratt zero-method (`zero_method='pratt'`) to conservatively retain zero-difference pairs in the Wilcoxon signed-rank distribution.*

### 5.2 Dimension 2: Conditional Integration Friction (Log-Normal Parametric AFT Model)
- **Conditional Evaluation**: Because our objective is to measure the friction of code that successfully reaches production, we evaluate merge latency conditionally on the PR ultimately being merged.
- **Log-Normal AFT Formulation**: Because PR merge latency is strictly positive and heavily right-skewed, we utilize a Log-Normal Parametric Regression (Accelerated Failure Time model) for our official Dimension 2 evaluation. This model estimates the time-delay multiplier (Time Ratio, $\text{TR} = \exp(\beta)$) of integrating infrastructure, controlling for baseline file volume (`log_app_count`), code churn (`log_app_line_churn`), test verification (`has_tests`), and repository popularity (`log_stars`).
- *Methodological Note on Clustered Sandwich Errors: To account for the paired dependence structure induced by 1:1 matching, we compute Huber-White cluster-robust sandwich covariance matrices clustered on matched pair_id.*
- *Numerical Stability Note on Duration Clipping: The outcome variable `duration` is clipped at lower bound $0.001$ hours (3.6 seconds) to prevent numerical divergence of $\log(0)$ for instantaneous merges in the Accelerated Failure Time model.*

In [13]:
# ==============================================================================
# SECTION 8: CAUSAL EVALUATION — DEEP MULTIDIMENSIONAL VELOCITY (RQ2)
# ==============================================================================
import numpy as np
import pandas as pd
from scipy import stats

print("=== 8. Running Deep Multidimensional Velocity Friction Analysis ===")

# --- Helper: Non-parametric Cliff's Delta Effect Size ---
def compute_cliffs_delta(treatment: np.ndarray, control: np.ndarray) -> float:
    """Computes non-parametric Cliff's Delta (-1 to +1) for paired/unpaired data."""
    n_t, n_c = len(treatment), len(control)
    diff_matrix = np.subtract.outer(treatment, control)
    delta = (np.sum(diff_matrix > 0) - np.sum(diff_matrix < 0)) / (n_t * n_c)
    return float(delta)


# ------------------------------------------------------------------------------
# 1. Macro Causal Latency Impact & Wilcoxon Significance (Dimension 1)
# ------------------------------------------------------------------------------
print("\n" + "=" * 66)
print("=== 5.1 DIMENSION 1: MACRO CAUSAL LATENCY IMPACT & WILCOXON SIGNIFICANCE ===")
print("=" * 66)

def run_macro_velocity_att(df_matched: pd.DataFrame, dataset_name: str):
    paired = df_matched.pivot(
        index="pair_id", 
        columns="match_role", 
        values="merge_latency"
    )
    diff = paired["Treatment"] - paired["Control"]
    treat = df_matched[df_matched["match_role"] == "Treatment"]["merge_latency"]
    ctrl = df_matched[df_matched["match_role"] == "Control"]["merge_latency"]
    
    # Wilcoxon signed-rank test on paired differences
    stat, p_val = stats.wilcoxon(diff, zero_method='pratt')
    delta = compute_cliffs_delta(treat.values, ctrl.values)
    
    print(f"\n[{dataset_name}]")
    print(f"Sample: {len(paired):,} Matched Pairs")
    print(f" - Median Paired Latency Diff : {diff.median():+.2f} hours")
    print(f" - Difference of Medians       : {treat.median() - ctrl.median():+.2f} hours (Treat: {treat.median():.2f}h vs Ctrl: {ctrl.median():.2f}h)")
    print(f" - Mean Paired Latency Diff   : {diff.mean():+.2f} hours (Treat: {treat.mean():.2f}h vs Ctrl: {ctrl.mean():.2f}h)")
    print(f" - Wilcoxon Signed-Rank Test   : W={stat:.1f}, p-value={p_val:.4e} {'(Statistically Significant)' if p_val < 0.05 else '(Not Significant)'}")
    print(f" - Cliff's Delta Effect Size   : delta={delta:+.3f} (Negligible < 0.147, Small < 0.33, Medium < 0.474)")

run_macro_velocity_att(df_aidev_matched, "AIDev v5")
run_macro_velocity_att(df_mosaic_matched, "MOSAIC-3M (Agentic Only)")

# ------------------------------------------------------------------------------
# 2. Conditional Integration Friction (Log-Normal Regression) (Dimension 2)
# ------------------------------------------------------------------------------
print("\n" + "=" * 66)
print("=== 5.2 DIMENSION 2: CONDITIONAL INTEGRATION FRICTION (LOG-NORMAL AFT MODEL) ===")
print("=" * 66)

try:
    from lifelines import LogNormalAFTFitter
    from autograd import grad
    from lifelines import utils

    # Enable cluster_col support in LogNormalAFTFitter to account for 1:1 matched pair clustering
    _orig_fit = LogNormalAFTFitter.fit
    _orig_compute_sandwich = LogNormalAFTFitter._compute_sandwich_errors

    def _clustered_fit(self, df, duration_col, event_col=None, cluster_col=None, robust=True, **kwargs):
        if cluster_col is not None:
            self.cluster_col = cluster_col
            self._clusters = df[cluster_col].values
            df = df.drop(columns=[cluster_col])
        else:
            self.cluster_col = None
            self._clusters = None
        return _orig_fit(self, df, duration_col=duration_col, event_col=event_col, robust=robust, **kwargs)

    def _clustered_compute_sandwich(self, Ts, E, weights, entries, Xs):
        if getattr(self, "cluster_col", None) is None:
            return _orig_compute_sandwich(self, Ts, E, weights, entries, Xs)
        with np.errstate(all="ignore"):
            ll_gradient = grad(self._neg_likelihood_with_penalty_function)
            params = self.params_.values
            scores = []
            for ts, e, w, s, (_, xs) in zip(utils.safe_zip(*Ts), E, weights, entries, Xs.iterrows()):
                xs_slice = utils.DataframeSlicer(xs.to_frame().T)
                scores.append(ll_gradient(params, ts, e, w, s, xs_slice))
            df_scores = pd.DataFrame(scores, index=self._clusters)
            cluster_scores = df_scores.groupby(level=0).sum().values
            J = cluster_scores.T @ cluster_scores
            return self.variance_matrix_.values @ J @ self.variance_matrix_.values

    LogNormalAFTFitter.fit = _clustered_fit
    LogNormalAFTFitter._compute_sandwich_errors = _clustered_compute_sandwich

    # Dimension 2 Evaluation: Parametric Log-Normal Regression Model
    print("\n" + "-" * 66)
    print("  EVALUATION: LOG-NORMAL PARAMETRIC REGRESSION MODEL")
    print("-" * 66)

    def fit_aft_model(df_matched: pd.DataFrame, dataset_name: str):
        aft_data = df_matched.copy()
        aft_data["duration"] = aft_data["merge_latency"].clip(lower=0.001)
        aft_data["event"] = 1  # All evaluated PRs are merged
        aft_data["is_treatment"] = (aft_data["match_role"] == "Treatment").astype(int)
        
        feature_cols = ["pair_id", "duration", "event", "is_treatment", "log_app_count", "log_app_line_churn", "has_tests", "log_stars"]
        clean_df = aft_data[feature_cols].dropna().copy()
        clean_df["has_tests"] = clean_df["has_tests"].astype(int)
        
        aft = LogNormalAFTFitter()
        aft.fit(clean_df, duration_col="duration", event_col="event", robust=True, cluster_col="pair_id")
        
        coef = aft.params_['mu_']['is_treatment']
        tr = np.exp(coef)
        p_val = aft.summary.loc[('mu_', 'is_treatment'), 'p']
        ci_lower = aft.summary.loc[('mu_', 'is_treatment'), 'exp(coef) lower 95%']
        ci_upper = aft.summary.loc[('mu_', 'is_treatment'), 'exp(coef) upper 95%']
        
        print(f"\n[{dataset_name} Log-Normal Parametric Regression Summary]")
        print(f" * Treatment Time Ratio (TR): {tr:.3f} (95% CI: [{ci_lower:.3f}, {ci_upper:.3f}], p={p_val:.4e})")
        
        if tr > 1.0:
            print(f"   => Dual-scope infrastructure modification INCREASES time-to-merge by {(tr - 1)*100:.1f}%.")
        else:
            print(f"   => Dual-scope infrastructure modification DECREASES time-to-merge by {(1 - tr)*100:.1f}%.")
            
        print("\nFull Model Summary Table:")
        cols_to_print = ['coef', 'exp(coef)', 'se(coef)', 'exp(coef) lower 95%', 'exp(coef) upper 95%', 'p']
        print(aft.summary[cols_to_print].to_string())

    fit_aft_model(df_aidev_matched, "AIDev v5")
    fit_aft_model(df_mosaic_matched, "MOSAIC-3M (Agentic Only)")

except ImportError:
    print("\n[NOTE] 'lifelines' is not currently installed. To compute Log-Normal Parametric Regression, run:")
    print("       !pip install lifelines")
    print("       Then re-run this cell.")


=== 8. Running Deep Multidimensional Velocity Friction Analysis ===

=== 5.1 DIMENSION 1: MACRO CAUSAL LATENCY IMPACT & WILCOXON SIGNIFICANCE ===

[AIDev v5]
Sample: 687 Matched Pairs
 - Median Paired Latency Diff : +0.84 hours
 - Difference of Medians       : +13.13 hours (Treat: 16.60h vs Ctrl: 3.47h)
 - Mean Paired Latency Diff   : +59.72 hours (Treat: 110.82h vs Ctrl: 51.11h)
 - Wilcoxon Signed-Rank Test   : W=80323.0, p-value=3.5462e-13 (Statistically Significant)
 - Cliff's Delta Effect Size   : delta=+0.202 (Negligible < 0.147, Small < 0.33, Medium < 0.474)

[MOSAIC-3M (Agentic Only)]
Sample: 80 Matched Pairs
 - Median Paired Latency Diff : -0.04 hours
 - Difference of Medians       : -0.72 hours (Treat: 2.21h vs Ctrl: 2.93h)
 - Mean Paired Latency Diff   : -35.98 hours (Treat: 48.71h vs Ctrl: 84.69h)
 - Wilcoxon Signed-Rank Test   : W=1568.0, p-value=8.0305e-01 (Not Significant)
 - Cliff's Delta Effect Size   : delta=-0.041 (Negligible < 0.147, Small < 0.33, Medium < 0.474)

==


------------------------------------------------------------------
  EVALUATION: LOG-NORMAL PARAMETRIC REGRESSION MODEL
------------------------------------------------------------------



[AIDev v5 Log-Normal Parametric Regression Summary]
 * Treatment Time Ratio (TR): 3.237 (95% CI: [2.545, 4.116], p=9.5572e-22)
   => Dual-scope infrastructure modification INCREASES time-to-merge by 223.7%.

Full Model Summary Table:
                               coef  exp(coef)  se(coef)  exp(coef) lower 95%  exp(coef) upper 95%             p
param  covariate                                                                                                
mu_    has_tests           1.440569   4.223099  0.215835             2.766382             6.446891  2.482454e-11
       is_treatment        1.174529   3.236617  0.122582             2.545361             4.115601  9.557204e-22
       log_app_count       0.255978   1.291724  0.097590             1.066845             1.564006  8.715976e-03
       log_app_line_churn  0.360882   1.434594  0.056948             1.283083             1.603995  2.342267e-10
       log_stars           0.536094   1.709317  0.044741             1.565810          


[MOSAIC-3M (Agentic Only) Log-Normal Parametric Regression Summary]
 * Treatment Time Ratio (TR): 0.794 (95% CI: [0.377, 1.672], p=5.4359e-01)
   => Dual-scope infrastructure modification DECREASES time-to-merge by 20.6%.

Full Model Summary Table:
                               coef  exp(coef)  se(coef)  exp(coef) lower 95%  exp(coef) upper 95%             p
param  covariate                                                                                                
mu_    has_tests           1.548690   4.705302  0.477721             1.844807            12.001183  1.187641e-03
       is_treatment       -0.230709   0.793971  0.379838             0.377125             1.671565  5.435940e-01
       log_app_count      -0.010615   0.989441  0.303641             0.545670             1.794112  9.721122e-01
       log_app_line_churn  0.305177   1.356865  0.132728             1.046065             1.760008  2.148944e-02
       log_stars           0.515313   1.674163  0.106008             1.3

### 5.3 Dimension 3: Subdomain Decomposed Parametric Survival Model

In Section 5.2 (Dimension 2), the Log-Normal Parametric Regression established that dual-scope infrastructure modification increases merge latency by $\text{TR} = 3.237$ ($p < 10^{-21}$). However, dual-scope PRs are heavily skewed toward CI/CD manifests (77.4%), with smaller shares for Automation (15.9%), Containers (15.7%), and Provisioning (8.9%). To test whether the $3.24\times$ latency multiplier is a universal infrastructure penalty or driven by specific operational bottlenecks (e.g., procedural automation vs. declarative pipelines), we execute an Infrastructure Subdomain Decomposed Survival Model.

We decompose the treatment effect into 4 additive indicators referencing the matched `App_Only` control group ($N = 687$ matched pairs, complete cases):
- `has_ci_cd`: CI/CD manifests (e.g., GitHub Actions, workflows).
- `has_automation`: Procedural build/automation scripts (e.g., Makefiles, Gradle, bash).
- `has_containers`: Containerization configs (e.g., Dockerfiles, Compose).
- `has_provisioning`: Cloud/IaC provisioning templates (e.g., Terraform, Kubernetes).

Covariates for file volume (`log_app_count`), code churn (`log_app_line_churn`), test verification (`has_tests`), and repository popularity (`log_stars`) are controlled, with robust Huber sandwich errors clustered on `pair_id`.

In [14]:
# ==============================================================================
# 8.1 SUBDOMAIN DECOMPOSED PARAMETRIC SURVIVAL MODEL (DIMENSION 3 EXTENDED)
# ==============================================================================
print("=== 5.3 SUBDOMAIN DECOMPOSED PARAMETRIC SURVIVAL MODEL ===")

# --------------------------------------------------------------------------
# 1. Feature Engineering & Category Mapping
# --------------------------------------------------------------------------
infra_files = df_commit_dedup[df_commit_dedup["is_infra"] == True].copy()

# PR-level indicator sets for Treatment PRs
ci_cd_prs = set(infra_files[infra_files["infra_category"] == "CI_CD"]["pr_id"])
containers_prs = set(infra_files[infra_files["infra_category"] == "Containers"]["pr_id"])
automation_prs = set(infra_files[infra_files["infra_category"] == "Automation"]["pr_id"])
provisioning_prs = set(infra_files[infra_files["infra_category"] == "Provisioning"]["pr_id"])

# Category mapping on df_aidev_matched
# If Control: exactly 0; If Treatment: check membership in category sets
is_treat = df_aidev_matched["match_role"] == "Treatment"

has_ci_cd = (is_treat & df_aidev_matched["id"].isin(ci_cd_prs)).astype(int)
has_containers = (is_treat & df_aidev_matched["id"].isin(containers_prs)).astype(int)
has_automation = (is_treat & df_aidev_matched["id"].isin(automation_prs)).astype(int)
has_provisioning = (is_treat & df_aidev_matched["id"].isin(provisioning_prs)).astype(int)

# --------------------------------------------------------------------------
# 2. Dataset Preparation for Survival Regression
# --------------------------------------------------------------------------
df_aft_subdomain = pd.DataFrame({
    "pair_id": df_aidev_matched["pair_id"],
    "merge_latency": df_aidev_matched["merge_latency"],
    "has_ci_cd": has_ci_cd,
    "has_containers": has_containers,
    "has_automation": has_automation,
    "has_provisioning": has_provisioning,
    "log_app_count": df_aidev_matched["log_app_count"],
    "log_app_line_churn": df_aidev_matched["log_app_line_churn"],
    "has_tests": df_aidev_matched["has_tests"].astype(int),
    "log_stars": df_aidev_matched["log_stars"],
})

# Drop any missing values
df_aft_subdomain = df_aft_subdomain.dropna().copy()

# Create target columns
df_aft_subdomain["duration"] = df_aft_subdomain["merge_latency"].clip(lower=0.001)
df_aft_subdomain["event"] = 1

reg_cols = [
    "pair_id", "duration", "event",
    "has_ci_cd", "has_containers", "has_automation", "has_provisioning",
    "log_app_count", "log_app_line_churn", "has_tests", "log_stars"
]
model_data = df_aft_subdomain[reg_cols].copy()

# --------------------------------------------------------------------------
# 3. Model Estimation (LogNormalAFTFitter with Clustered Sandwich Errors)
# --------------------------------------------------------------------------
aft_subdomain = LogNormalAFTFitter()
aft_subdomain.fit(
    model_data,
    duration_col="duration",
    event_col="event",
    robust=True,
    cluster_col="pair_id"
)

# --------------------------------------------------------------------------
# 4. Reporting Requirements
# --------------------------------------------------------------------------
# a. Print descriptive category counts in matched Treatment PRs
n_treat = (df_aidev_matched["match_role"] == "Treatment").sum()
treat_df = df_aft_subdomain.loc[df_aidev_matched["match_role"] == "Treatment"]

print(f"\n--- Matched Treatment PR Infrastructure Subdomain Distribution (N = {n_treat:,}) ---")
cat_info = [
    ("CI/CD Manifests (`has_ci_cd`)", treat_df["has_ci_cd"].sum()),
    ("Automation Scripts (`has_automation`)", treat_df["has_automation"].sum()),
    ("Containerization (`has_containers`)", treat_df["has_containers"].sum()),
    ("System Provisioning (`has_provisioning`)", treat_df["has_provisioning"].sum()),
]
for cat_name, count in cat_info:
    pct = (count / n_treat) * 100
    print(f" * {cat_name:<42}: {count:>3} PRs ({pct:>5.1f}%)")

# b. Display Full AFT Regression Summary Table
print("\n" + "=" * 80)
print("  FULL LOG-NORMAL AFT REGRESSION SUMMARY TABLE (SUBDOMAIN DECOMPOSED)")
print("=" * 80)
summary_cols = ["coef", "exp(coef)", "se(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]
print(aft_subdomain.summary[summary_cols].round(4).to_string())

# c. Display Infrastructure Category Bottleneck Summary Table
categories = [
    ("CI/CD Manifests (`has_ci_cd`)", "has_ci_cd", treat_df["has_ci_cd"].sum()),
    ("Automation Scripts (`has_automation`)", "has_automation", treat_df["has_automation"].sum()),
    ("Containerization (`has_containers`)", "has_containers", treat_df["has_containers"].sum()),
    ("System Provisioning (`has_provisioning`)", "has_provisioning", treat_df["has_provisioning"].sum()),
]

bottleneck_rows = []
for display_name, col_name, pr_count in categories:
    row_data = aft_subdomain.summary.loc[("mu_", col_name), :]
    tr = row_data["exp(coef)"]
    ci_lower = row_data["exp(coef) lower 95%"]
    ci_upper = row_data["exp(coef) upper 95%"]
    p_val = row_data["p"]
    
    multiplier = (tr - 1.0) * 100.0 if tr > 1.0 else -(1.0 - tr) * 100.0
    
    if p_val < 0.001 and tr >= 2.5:
        severity = "Severe Bottleneck"
    elif p_val < 0.05 and tr >= 1.5:
        severity = "Moderate Bottleneck"
    elif p_val < 0.05 and tr < 1.0:
        severity = "Latency Reduction (Significant)"
    else:
        severity = "Not Statistically Significant"
        
    bottleneck_rows.append({
        "Infrastructure Subdomain": display_name,
        "Matched Treatment PRs": f"{pr_count} ({(pr_count / n_treat) * 100:.1f}%)",
        "Time Ratio (exp(coef))": f"{tr:.3f}",
        "95% CI": f"[{ci_lower:.3f}, {ci_upper:.3f}]",
        "Latency Multiplier (%)": f"{multiplier:+.1f}%",
        "p-value": f"{p_val:.4e}" if p_val < 0.0001 else f"{p_val:.4f}",
        "Bottleneck Severity": severity
    })

df_bottleneck = pd.DataFrame(bottleneck_rows)
print("\n" + "=" * 90)
print("  INFRASTRUCTURE SUBDOMAIN BOTTLENECK SEVERITY TABLE")
print("=" * 90)
print(df_bottleneck.to_string(index=False))


=== 5.3 SUBDOMAIN DECOMPOSED PARAMETRIC SURVIVAL MODEL ===



--- Matched Treatment PR Infrastructure Subdomain Distribution (N = 687) ---
 * CI/CD Manifests (`has_ci_cd`)             : 532 PRs ( 77.4%)
 * Automation Scripts (`has_automation`)     : 109 PRs ( 15.9%)
 * Containerization (`has_containers`)       : 108 PRs ( 15.7%)
 * System Provisioning (`has_provisioning`)  :  61 PRs (  8.9%)

  FULL LOG-NORMAL AFT REGRESSION SUMMARY TABLE (SUBDOMAIN DECOMPOSED)
                             coef  exp(coef)  se(coef)  exp(coef) lower 95%  exp(coef) upper 95%       p
param  covariate                                                                                        
mu_    has_automation      0.4199     1.5218    0.3088               0.8308               2.7874  0.1739
       has_ci_cd           1.3421     3.8272    0.1384               2.9177               5.0203  0.0000
       has_containers      0.1239     1.1319    0.2791               0.6550               1.9560  0.6571
       has_provisioning    1.0033     2.7274    0.3720               1

### 5.4 Cross-Dataset Infrastructure Subdomain Latency Differentials

To contextualize the parametric hazard decomposition, we measure the empirical median merge latency across infrastructure subdomains (`CI_CD`, `Containers`, `Automation`, `Provisioning`) in the dual-scope population across both AIDev v5 and MOSAIC-3M cohorts.

In [15]:
# ==============================================================================
# 3. Contextual Bottlenecks: Infrastructure Subdomain Differentials (Dimension 3)
# ==============================================================================
print("=" * 66)
print("=== 5.4 DIMENSION 3: CONTEXTUAL BOTTLENECKS BY INFRASTRUCTURE SUBDOMAIN ===")
print("=" * 66)
print("\n[Cross-Dataset Median Merge Latency by Infrastructure Category (App_and_Infra)]")
print(comp_infra.to_string())


=== 5.4 DIMENSION 3: CONTEXTUAL BOTTLENECKS BY INFRASTRUCTURE SUBDOMAIN ===

[Cross-Dataset Median Merge Latency by Infrastructure Category (App_and_Infra)]
                AIDev PRs  AIDev Share (%)  AIDev Median Latency (h)  MOSAIC PRs  MOSAIC Share (%)  MOSAIC Median Latency (h)
infra_category                                                                                                               
CI_CD                 755            61.13                     39.29         144             62.07                       4.15
Automation            199            16.11                     78.26          27             11.64                      26.21
Containers            184            14.90                     32.04          43             18.53                       4.10
Provisioning           97             7.85                     32.89          18              7.76                      24.51


## Section 6: Treatment Integrity & Structural Dual-Scope Diagnostics

To ensure internal and construct validity of the treatment condition (`App_and_Infra`), we evaluate two key structural diagnostics across our matched cohorts:

### 6.1 Action Integrity in Matched Treatment Units (Active Authoring vs. Pure Deletions)
Distinguishing whether infrastructure modifications represent active authoring (`infra_active_count > 0`) versus pure file deletions / retirements (`infra_deleted_count > 0` with zero active modifications).

### 6.2 Strict vs. Weak Dual-Scope Latency Penalty (Core Logic vs. Configs)
Partitioning Treatment PRs into **Strict Dual-Scope** (modifying functional application logic `sub_type == 'App'`), **Weak Dual-Scope** (modifying application build/package manifests `sub_type == 'App_Config'` without application logic), and **Other Dual-Scope**, evaluated against the matched control baseline.

In [16]:
# ==============================================================================
# 9. DIAGNOSTIC: ACTION INTEGRITY AND DUAL-SCOPE NUANCE IN MATCHED COHORTS
# ==============================================================================
print("=== 9. Diagnostic: Action Integrity and Dual-Scope Nuance in Matched Cohorts ===")

# ------------------------------------------------------------------------------
# 1. Action Integrity in Matched Cohorts (Treatment Units)
# ------------------------------------------------------------------------------
print("\n" + "=" * 66)
print("=== 6.1 DIAGNOSTIC: ACTION INTEGRITY IN MATCHED TREATMENT UNITS ===")
print("=" * 66)

# Filter for Treatment PRs
aidev_treat = df_aidev_matched[df_aidev_matched["match_role"] == "Treatment"].copy()
mosaic_treat = df_mosaic_matched[df_mosaic_matched["match_role"] == "Treatment"].copy()

# A. AIDev v5 Treatment PRs
aidev_treat_files = aidev_treat[["id", "merge_latency"]].merge(
    df_commit_dedup[["pr_id", "filename", "status", "changes"]],
    left_on="id",
    right_on="pr_id",
    how="inner"
)

aidev_file_classes = [
    classify_file_domain(fn, st, ch)
    for fn, st, ch in zip(aidev_treat_files["filename"], aidev_treat_files["status"], aidev_treat_files["changes"])
]
aidev_treat_files["is_infra"] = [c.get("is_infra", False) for c in aidev_file_classes]
aidev_treat_files["is_infra_deleted"] = [c.get("is_infra_deleted", False) for c in aidev_file_classes]
aidev_treat_files["sub_type"] = [c.get("sub_type") for c in aidev_file_classes]

aidev_treat_files["is_infra_active"] = aidev_treat_files["is_infra"] & (~aidev_treat_files["is_infra_deleted"])
aidev_treat_files["is_infra_del"] = aidev_treat_files["is_infra"] & (aidev_treat_files["is_infra_deleted"])

aidev_pr_infra = aidev_treat_files.groupby("id").agg(
    infra_active_count=("is_infra_active", "sum"),
    infra_deleted_count=("is_infra_del", "sum")
).reset_index()

def categorize_action_integrity(row):
    if row["infra_active_count"] > 0:
        return "Active Authoring"
    elif row["infra_active_count"] == 0 and row["infra_deleted_count"] > 0:
        return "Pure Deletion"
    else:
        return "Other"

aidev_pr_infra["action_category"] = aidev_pr_infra.apply(categorize_action_integrity, axis=1)

# B. MOSAIC-3M Treatment PRs
if "files" not in mosaic_treat.columns:
    mosaic_treat = mosaic_treat.merge(df_mosaic_pr_filtered[["id", "files"]], on="id", how="left")

mosaic_files_parsed = mosaic_treat["files"].apply(parse_mosaic_files_array)
mosaic_treat["infra_active_count"] = mosaic_files_parsed["infra_active_count"]
mosaic_treat["infra_deleted_count"] = mosaic_files_parsed["infra_deleted_count"]
mosaic_treat["action_category"] = mosaic_treat.apply(categorize_action_integrity, axis=1)

# Summary table
aidev_counts = aidev_pr_infra["action_category"].value_counts()
aidev_pcts = (aidev_pr_infra["action_category"].value_counts(normalize=True) * 100)

mosaic_counts = mosaic_treat["action_category"].value_counts()
mosaic_pcts = (mosaic_treat["action_category"].value_counts(normalize=True) * 100)

categories = ["Active Authoring", "Pure Deletion"]
summary_rows = []
for cat in categories:
    a_cnt = int(aidev_counts.get(cat, 0))
    a_pct = float(aidev_pcts.get(cat, 0.0))
    m_cnt = int(mosaic_counts.get(cat, 0))
    m_pct = float(mosaic_pcts.get(cat, 0.0))
    summary_rows.append({
        "Action Category": cat,
        "AIDev PRs": a_cnt,
        "AIDev Share (%)": f"{a_pct:.2f}%",
        "MOSAIC PRs": m_cnt,
        "MOSAIC Share (%)": f"{m_pct:.2f}%"
    })

df_action_summary = pd.DataFrame(summary_rows)
print("\n[Action Integrity in Matched Treatment Units: Active Authoring vs. Pure Deletion]")
print(df_action_summary.to_string(index=False))

# ------------------------------------------------------------------------------
# 2. Strict vs. Weak Dual-Scope Latency Penalty (AIDev v5 Only)
# ------------------------------------------------------------------------------
print("\n" + "=" * 66)
print("=== 6.2 DIAGNOSTIC: STRICT VS. WEAK DUAL-SCOPE LATENCY PENALTY ===")
print("=" * 66)

sub_type_sets = aidev_treat_files.groupby("id")["sub_type"].agg(set)
def categorize_from_set(s):
    if "App" in s:
        return "Strict Dual-Scope"
    elif "App_Config" in s:
        return "Weak Dual-Scope"
    else:
        return "Other Dual-Scope"

aidev_dual_scope = sub_type_sets.apply(categorize_from_set).reset_index(name="dual_scope_bucket")
aidev_treat_scoped = aidev_treat[["id", "merge_latency"]].merge(aidev_dual_scope, on="id", how="inner")

bucket_order = ["Strict Dual-Scope", "Weak Dual-Scope", "Other Dual-Scope"]
dual_scope_summary = []

for b in bucket_order:
    sub = aidev_treat_scoped[aidev_treat_scoped["dual_scope_bucket"] == b]
    cnt = len(sub)
    share = (cnt / len(aidev_treat_scoped)) * 100 if len(aidev_treat_scoped) > 0 else 0.0
    med_lat = sub["merge_latency"].median() if cnt > 0 else np.nan
    dual_scope_summary.append({
        "Dual-Scope Bucket": b,
        "Treatment PRs": cnt,
        "Share (%)": f"{share:.2f}%",
        "Median Latency (h)": f"{med_lat:.2f}" if not np.isnan(med_lat) else "N/A"
    })

df_dual_scope_summary = pd.DataFrame(dual_scope_summary)
print("\n[AIDev v5 Treatment Latency by Dual-Scope Subtype]")
print(df_dual_scope_summary.to_string(index=False))

# Baseline comparison: overall AIDev Control group
aidev_control = df_aidev_matched[df_aidev_matched["match_role"] == "Control"]
ctrl_count = len(aidev_control)
ctrl_median = aidev_control["merge_latency"].median()

print(f"\n[Baseline Comparison: AIDev v5 Matched Control Group (App-Only)]")
print(f" * Control PR Count       : {ctrl_count:,}")
print(f" * Baseline Median Latency: {ctrl_median:.2f} hours")


=== 9. Diagnostic: Action Integrity and Dual-Scope Nuance in Matched Cohorts ===

=== 6.1 DIAGNOSTIC: ACTION INTEGRITY IN MATCHED TREATMENT UNITS ===



[Action Integrity in Matched Treatment Units: Active Authoring vs. Pure Deletion]
 Action Category  AIDev PRs AIDev Share (%)  MOSAIC PRs MOSAIC Share (%)
Active Authoring        663          96.51%          80          100.00%
   Pure Deletion         24           3.49%           0            0.00%

=== 6.2 DIAGNOSTIC: STRICT VS. WEAK DUAL-SCOPE LATENCY PENALTY ===

[AIDev v5 Treatment Latency by Dual-Scope Subtype]
Dual-Scope Bucket  Treatment PRs Share (%) Median Latency (h)
Strict Dual-Scope            617    89.81%              19.64
  Weak Dual-Scope             70    10.19%               1.13
 Other Dual-Scope              0     0.00%                N/A

[Baseline Comparison: AIDev v5 Matched Control Group (App-Only)]
 * Control PR Count       : 687
 * Baseline Median Latency: 3.47 hours
